In [1]:
#!/usr/bin/env python3
"""
01_preprocess_rna_atac.py
=========================
Generates:
  - scenicplus_input/rna_scanpy_preprocessed.h5ad
  - scenicplus_input/atac_cistopic_selected_model_chrstr.pkl
  - scenicplus_input/region_sets_topics_otsu.pkl

Consumes outputs of convert_combined_to_scenic.R:
  - scenicplus_input/rna_matrix.mtx
  - scenicplus_input/rna_features.tsv
  - scenicplus_input/rna_barcodes.tsv
  - scenicplus_input/rna_cell_metadata.csv
  - scenicplus_input/atac_matrix.mtx
  - scenicplus_input/atac_barcodes.tsv
  - scenicplus_input/peaks.bed
  - scenicplus_input/atac_cell_metadata.csv
  - scenicplus_input/umap_combined.csv
"""

import os, json, warnings, pickle, gc, random
import numpy as np
import pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix
import anndata as ad
import scanpy as sc

# ============================================================
# CONFIG
# ============================================================
INDIR = "scenicplus_input"

ALREADY_RAN = True 

if(not ALREADY_RAN):
    OUT_RNA_H5AD      = os.path.join(INDIR, "rna_scanpy_preprocessed.h5ad")
    OUT_CISTOPIC_PKL  = os.path.join(INDIR, "atac_cistopic_selected_model_chrstr.pkl")
    OUT_BIN_TOPICS    = os.path.join(INDIR, "region_sets_topics_otsu.pkl")
    
    # RNA preprocessing
    MIN_GENES        = 200
    MIN_COUNTS       = 500
    MAX_GENES        = 7500
    N_HVG            = 3000
    TARGET_SUM       = 1e4
    N_PCS            = 50
    N_NEIGH          = 15
    N_PCS_NEIGH      = 30
    LEIDEN_RES       = 0.6
    
    # ATAC / cisTopic
    TARGET_MAX_PEAKS          = 50_000   # filter peaks to at most this many
    FALLBACK_MIN_CELLS_PER_PEAK = 80     # hard floor on cells-per-peak filter
    MAX_ATAC_CELLS            = 15_000   # downsample ATAC cells for LDA if needed
    RNG_SEED                  = 555
    
    # LDA
    TOPICS_TO_TRY  = [10, 12]
    N_ITER         = 120
    RANDOM_STATE   = 555
    N_CPU          = 1
    RAY_LOCAL_MODE = True   # avoids Ray object-store OOM on WSL2/laptops
    
    # ============================================================
    # HELPERS
    # ============================================================
    def _make_unique(idx) -> pd.Index:
        """Make a pandas Index unique by appending -1, -2, ... suffixes."""
        idx = pd.Index(pd.Series(idx, dtype=str))
        if idx.is_unique:
            return idx
        seen, out = {}, []
        for x in idx:
            seen[x] = seen.get(x, -1) + 1
            out.append(x if seen[x] == 0 else f"{x}-{seen[x]}")
        return pd.Index(out)
    
    
    def sanitize_for_h5ad(adata: ad.AnnData) -> ad.AnnData:
        """Coerce obs/var columns to plain str so h5ad write doesn't choke."""
        for df in (adata.obs, adata.var):
            for col in list(df.columns):
                s = df[col]
                if hasattr(s.dtype, "categories") or s.dtype == object:
                    df[col] = s.astype(str)
        return adata
    
    
    # ============================================================
    # STEP 1 — RNA PREPROCESSING
    # ============================================================
    print("=" * 60)
    print("STEP 1: RNA preprocessing")
    print("=" * 60)
    
    # --- Load MTX ---
    mtx_path   = os.path.join(INDIR, "rna_matrix.mtx")
    feats_path = os.path.join(INDIR, "rna_features.tsv")
    bcs_path   = os.path.join(INDIR, "rna_barcodes.tsv")
    meta_path  = os.path.join(INDIR, "rna_cell_metadata.csv")
    
    X        = mmread(mtx_path).tocsr()
    features = pd.read_csv(feats_path, sep="\t", header=None)
    barcodes = pd.read_csv(bcs_path,   sep="\t", header=None).iloc[:, 0].astype(str).tolist()
    
    if features.shape[1] >= 2:
        features.columns = ["gene_id", "gene_name"] + list(features.columns[2:])
        gene_names = features["gene_name"].fillna(features["gene_id"]).astype(str).tolist()
        var_df = pd.DataFrame(
            {"gene_id": features["gene_id"].astype(str).values,
             "gene_name": features["gene_name"].astype(str).values},
            index=_make_unique(gene_names)
        )
    else:
        gene_names = features.iloc[:, 0].astype(str).tolist()
        var_df = pd.DataFrame(index=_make_unique(gene_names))
    
    n_cells = len(barcodes)
    n_genes = len(gene_names)
    
    # Orient to cells × genes
    if X.shape == (n_genes, n_cells):
        X = X.T
        print(f"[RNA] Transposed matrix: {X.shape}")
    elif X.shape == (n_cells, n_genes):
        print(f"[RNA] Matrix already cells×genes: {X.shape}")
    else:
        raise ValueError(f"Matrix shape {X.shape} doesn't match "
                         f"({n_cells} cells, {n_genes} genes)")
    
    obs_df = pd.DataFrame(index=_make_unique(barcodes))
    rna = ad.AnnData(X=X, obs=obs_df, var=var_df)
    rna.var_names_make_unique()
    rna.obs_names_make_unique()
    print(f"[RNA] Loaded: {rna.n_obs:,} cells × {rna.n_vars:,} genes")
    
    # --- Attach metadata ---
    if os.path.exists(meta_path):
        meta = pd.read_csv(meta_path, low_memory=False)
        key  = "cell" if "cell" in meta.columns else meta.columns[0]
        meta = meta.set_index(key)
        meta.index = meta.index.astype(str)
        for c in meta.columns:
            if meta[c].dtype == object:
                meta[c] = meta[c].astype(str)
        rna.obs = rna.obs.join(meta, how="left")
        print(f"[RNA] Joined {len(meta.columns)} metadata columns")
    
    # --- Attach pre-computed UMAP from Seurat ---
    umap_path = os.path.join(INDIR, "umap_combined.csv")
    if os.path.exists(umap_path):
        umap_df = pd.read_csv(umap_path, low_memory=False)
        key = "cell" if "cell" in umap_df.columns else umap_df.columns[0]
        umap_df = umap_df.set_index(key).reindex(rna.obs_names)
        if {"UMAP_1", "UMAP_2"}.issubset(umap_df.columns):
            rna.obsm["X_umap"] = umap_df[["UMAP_1", "UMAP_2"]].to_numpy()
            print(f"[RNA] Attached UMAP from umap_combined.csv")
            # merge any extra metadata columns (e.g. celltype, seurat_clusters)
            extra = [c for c in umap_df.columns if c not in {"UMAP_1", "UMAP_2"}]
            for c in extra:
                if c not in rna.obs.columns:
                    rna.obs[c] = umap_df[c].astype(str).values
    
    # --- QC filters ---
    sc.pp.calculate_qc_metrics(rna, inplace=True)
    pre_cells = rna.n_obs
    sc.pp.filter_cells(rna, min_genes=MIN_GENES)
    sc.pp.filter_cells(rna, min_counts=MIN_COUNTS)
    rna = rna[rna.obs["n_genes_by_counts"] < MAX_GENES].copy()
    print(f"[RNA] After QC filters: {rna.n_obs:,} cells (removed {pre_cells - rna.n_obs:,})")
    
    # --- Normalize & HVG ---
    sc.pp.normalize_total(rna, target_sum=TARGET_SUM)
    sc.pp.log1p(rna)
    rna.raw = rna
    
    try:
        sc.pp.highly_variable_genes(rna, n_top_genes=N_HVG, flavor="seurat_v5")
    except Exception:
        sc.pp.highly_variable_genes(rna, n_top_genes=N_HVG, flavor="seurat_v3")
    
    n_hvg = int(rna.var["highly_variable"].sum())
    rna   = rna[:, rna.var["highly_variable"]].copy()
    print(f"[RNA] HVGs selected: {n_hvg:,}")
    
    # --- PCA / UMAP / Leiden only if no pre-computed UMAP ---
    if "X_umap" not in rna.obsm:
        print("[RNA] Computing PCA, neighbors, UMAP, Leiden...")
        sc.pp.scale(rna, max_value=10)
        sc.tl.pca(rna, n_comps=N_PCS, svd_solver="arpack")
        n_pcs_used = min(N_PCS_NEIGH, rna.obsm["X_pca"].shape[1])
        sc.pp.neighbors(rna, n_neighbors=N_NEIGH, n_pcs=n_pcs_used)
        sc.tl.umap(rna)
        sc.tl.leiden(rna, resolution=LEIDEN_RES)
    
    # --- Save ---
    rna = sanitize_for_h5ad(rna)
    rna.write(OUT_RNA_H5AD, compression="gzip")
    print(f"[RNA] Wrote: {OUT_RNA_H5AD}")
    print(f"[RNA] Final: {rna.n_obs:,} cells × {rna.n_vars:,} genes")
    
    del rna
    gc.collect()
    
    
    # ============================================================
    # STEP 2 — ATAC cisTopic
    # ============================================================
    print("\n" + "=" * 60)
    print("STEP 2: ATAC cisTopic")
    print("=" * 60)
    
    atac_mtx_path  = os.path.join(INDIR, "atac_matrix.mtx")
    atac_bcs_path  = os.path.join(INDIR, "atac_barcodes.tsv")
    peaks_bed_path = os.path.join(INDIR, "peaks.bed")
    atac_meta_path = os.path.join(INDIR, "atac_cell_metadata.csv")
    frag_json_path = os.path.join(INDIR, "fragments_paths.json")
    
    A          = mmread(atac_mtx_path).tocsr()   # rows=peaks, cols=cells
    atac_cells = pd.read_csv(atac_bcs_path, sep="\t", header=None).iloc[:, 0].astype(str).tolist()
    peaks_df   = pd.read_csv(peaks_bed_path, sep="\t", header=None, names=["chr", "start", "end"])
    
    print(f"[ATAC] Raw matrix: {A.shape[0]:,} peaks × {A.shape[1]:,} cells, nnz={A.nnz:,}")
    
    # --- Build region names in cisTopic/cisTarget format ---
    # Strip chr prefix → bare chromosome numbers → ":chrom:start-end" format
    # This matches the consensus_regions.fa headers (>::1:33501-34428)
    # and the cisTarget DB column names (1:33501-34428) after stripping >::
    chrom = peaks_df["chr"].astype(str).str.replace(r"^chr", "", regex=True)
    region_names = (
        ":" + chrom + ":" +
        peaks_df["start"].astype(int).astype(str) + "-" +
        peaks_df["end"].astype(int).astype(str)
    ).tolist()
    print(f"[ATAC] Region name sample: {region_names[:3]}")
    
    # --- Subset to RNA∩ATAC barcodes ---
    rna_reloaded = ad.read_h5ad(OUT_RNA_H5AD)
    common = list(pd.Index(atac_cells).intersection(rna_reloaded.obs_names))
    print(f"[ATAC] Common barcodes (RNA∩ATAC): {len(common):,}")
    del rna_reloaded
    gc.collect()
    
    use_cells = common if common else atac_cells
    if not common:
        warnings.warn("[ATAC] No common barcodes — using all ATAC cells (non-multiome mode)")
    
    # --- Optional cell downsampling ---
    if MAX_ATAC_CELLS and len(use_cells) > MAX_ATAC_CELLS:
        random.seed(RNG_SEED)
        use_cells = sorted(random.sample(use_cells, MAX_ATAC_CELLS))
        print(f"[ATAC] Downsampled to {len(use_cells):,} cells for LDA")
    
    cell_to_col = {c: i for i, c in enumerate(atac_cells)}
    cols_idx    = np.array([cell_to_col[c] for c in use_cells], dtype=int)
    A           = A[:, cols_idx]
    
    # --- Adaptive peak filter ---
    binary_A    = (A > 0).astype("int32").tocsr()
    peak_counts = np.asarray(binary_A.sum(axis=1)).ravel()
    n_before    = A.shape[0]
    
    if n_before > TARGET_MAX_PEAKS:
        rank = max(0, n_before - TARGET_MAX_PEAKS)
        thr  = max(int(np.partition(peak_counts, rank)[rank]),
                   int(FALLBACK_MIN_CELLS_PER_PEAK))
    else:
        thr = 0
    
    keep        = peak_counts >= thr
    A           = A[keep, :]
    binary_A    = binary_A[keep, :]
    region_names = [r for r, k in zip(region_names, keep) if k]
    peaks_df    = peaks_df.loc[keep].reset_index(drop=True)
    
    print(f"[ATAC] Peak filter (min_cells={thr}): {n_before:,} → {A.shape[0]:,} peaks")
    print(f"[ATAC] Using {A.shape[1]:,} cells, {A.shape[0]:,} peaks, nnz={A.nnz:,}")
    
    # Save peak stats for reference
    pd.DataFrame({
        "peaks_before": [n_before],
        "peaks_after":  [A.shape[0]],
        "cells":        [A.shape[1]],
        "nnz":          [A.nnz],
        "min_cells_per_peak": [thr]
    }).to_csv(os.path.join(INDIR, "atac_peak_stats.csv"), index=False)
    
    # --- Load cell metadata ---
    cell_meta = None
    if os.path.exists(atac_meta_path):
        md = pd.read_csv(atac_meta_path, index_col=0, low_memory=False)
        md.index = md.index.astype(str)
        for c in md.columns:
            if md[c].dtype == object:
                md[c] = md[c].astype(str)
        cell_meta = md.reindex(use_cells)
    
    # --- Fragments paths ---
    path_to_fragments = None
    if os.path.exists(frag_json_path):
        try:
            frag_list = json.load(open(frag_json_path)).get("fragments", []) or []
            if len(frag_list) > 1:
                path_to_fragments = {f"sample{i+1}": p for i, p in enumerate(frag_list)}
            elif len(frag_list) == 1:
                path_to_fragments = frag_list[0]
        except Exception:
            pass
    
    # --- Build CistopicObject ---
    from pycisTopic.cistopic_class import CistopicObject
    
    cistopic = CistopicObject(
        fragment_matrix  = A,
        binary_matrix    = binary_A,
        cell_names       = use_cells,
        region_names     = region_names,
        cell_data        = cell_meta,
        region_data      = peaks_df.copy(),
        path_to_fragments = path_to_fragments,
        project          = "cisTopic_GallusGallus"
    )
    del peak_counts
    gc.collect()
    
    # --- Run LDA models ---
    print("\n[ATAC] Running LDA topic models...")
    os.environ.setdefault("RAY_memory_usage_threshold", "0.99")
    from pycisTopic.lda_models import run_cgs_models, evaluate_models
    
    all_models = []
    for k in TOPICS_TO_TRY:
        print(f"  Fitting {k} topics (n_iter={N_ITER})...")
        models_k = run_cgs_models(
            cistopic_obj    = cistopic,
            n_topics        = [k],
            n_cpu           = N_CPU,
            n_iter          = N_ITER,
            random_state    = RANDOM_STATE,
            local_mode      = RAY_LOCAL_MODE,
            include_dashboard = False,
        )
        all_models.extend(models_k)
        gc.collect()
    
    best_model = evaluate_models(all_models, select_model=None, return_model=True)
    cistopic.add_LDA_model(best_model)
    cistopic.selected_model = best_model
    n_topics_best = getattr(best_model, "n_topics", None) or getattr(best_model, "K", "?")
    print(f"[ATAC] Selected model: {n_topics_best} topics")
    
    # --- Save cisTopic object ---
    # Named _chrstr because region names use the :chrom:start-end string format
    with open(OUT_CISTOPIC_PKL, "wb") as fh:
        pickle.dump(cistopic, fh, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"[ATAC] Saved: {OUT_CISTOPIC_PKL}")
    print(f"[ATAC] Region name format check: {cistopic.region_names[:3]}")
    
    # --- Binarize topics (Otsu) ---
    print("\n[ATAC] Binarizing topics (Otsu method)...")
    from pycisTopic.topic_binarization import binarize_topics
    
    region_sets = binarize_topics(
        cistopic_obj  = cistopic,
        target        = "region",
        method        = "otsu",
        smooth_topics = True,
        plot          = False
    )
    with open(OUT_BIN_TOPICS, "wb") as fh:
        pickle.dump(region_sets, fh)
    print(f"[ATAC] Wrote: {OUT_BIN_TOPICS}")
    
    if isinstance(region_sets, dict):
        for name, rs in sorted(region_sets.items()):
            pct = len(rs) / len(cistopic.region_names) * 100
            print(f"  {name}: {len(rs):,} regions ({pct:.1f}%)")
    
    print("\nDone. Ready to run make_genome_annots.py and the cisTarget DB build.")
    


In [2]:
# SCENIC+: core paths and imports
import warnings
import anndata
try:
    anndata.settings.verbosity = 0
except:
    pass

warnings.filterwarnings("ignore")

import os
from pathlib import Path
import re
import glob
import gc
import pickle

import numpy as np
import pandas as pd
import anndata as ad
import pyranges as pr

from scipy.io import mmread

# If scanpy / pycisTopic / scenicplus are installed in this env:
import scanpy as sc
from pycisTopic.cistopic_class import CistopicObject
from pycisTopic.topic_binarization import binarize_topics

print("Python imports OK.")

# ------------------------------------------------------------------
# PATHS – EDIT THIS BLOCK IF YOUR ROOT IS DIFFERENT
# ------------------------------------------------------------------
# Project root (parent of "scenicplus_input" and "Snakemake")
PROJECT_ROOT = Path("").resolve()

INDIR          = PROJECT_ROOT / "scenicplus_input"
SNAKEMAKE_DIR  = PROJECT_ROOT / "scplus_pipeline" / "Snakemake"  # adjust if layout differs
# If your Snakemake folder is actually PROJECT_ROOT / "Snakemake":
# SNAKEMAKE_DIR  = PROJECT_ROOT / "Snakemake"

# Core input/output files used by config.yaml
RNA_H5AD       = INDIR / "rna_scanpy_preprocessed.h5ad"
CISTOPIC_PKL   = INDIR / "atac_cistopic_selected_model_chrstr.pkl"
CISTOPIC_RAW   = INDIR / "atac_cistopic_selected_model.pkl"  # intermediate
REGION_SETS_PKL = INDIR / "region_sets_topics_otsu.pkl"
REGION_SETS_DIR = INDIR / "region_sets"

# cisTarget DBs (as referenced by config.yaml)
CTX_DB_RANKINGS = INDIR / "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather"
CTX_DB_SCORES   = INDIR / "gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather"

# Motif annotations
MOTIF_ANNOT_RAW = INDIR / "v10nr_clust_public" / "snapshots" / "motifs-v10-nr.chicken-m0.00001-o0.0.tbl"
MOTIF_ANNOT_FILT = INDIR / "motifs.filtered.tbl"  # what config.yaml points to

# Genome annotation (downloaded in Snakemake dir)
GENOME_ANNOT_TSV = SNAKEMAKE_DIR / "genome_annotation.tsv"

print("\n[PATHS]")
print("PROJECT_ROOT:       ", PROJECT_ROOT)
print("INDIR:              ", INDIR)
print("SNAKEMAKE_DIR:      ", SNAKEMAKE_DIR)
print("RNA_H5AD:           ", RNA_H5AD)
print("CISTOPIC_PKL:       ", CISTOPIC_PKL)
print("REGION_SETS_DIR:    ", REGION_SETS_DIR)
print("CTX_DB_RANKINGS:    ", CTX_DB_RANKINGS)
print("CTX_DB_SCORES:      ", CTX_DB_SCORES)
print("MOTIF_ANNOT_RAW:    ", MOTIF_ANNOT_RAW)
print("MOTIF_ANNOT_FILT:   ", MOTIF_ANNOT_FILT)
print("GENOME_ANNOT_TSV:   ", GENOME_ANNOT_TSV)

assert INDIR.exists(), f"Input directory not found: {INDIR}"


Python imports OK.

[PATHS]
PROJECT_ROOT:        /mnt/c/Users/jacwe/Downloads/SAMAP
INDIR:               /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input
SNAKEMAKE_DIR:       /mnt/c/Users/jacwe/Downloads/SAMAP/scplus_pipeline/Snakemake
RNA_H5AD:            /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/rna_scanpy_preprocessed.h5ad
CISTOPIC_PKL:        /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/atac_cistopic_selected_model_chrstr.pkl
REGION_SETS_DIR:     /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets
CTX_DB_RANKINGS:     /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
CTX_DB_SCORES:       /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather
MOTIF_ANNOT_RAW:     /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/v10nr_clust_public/snapshots/motifs-v10-nr.chicken-m0.00001-o0.0.tbl
MOTIF_ANNOT_FILT:    /mnt/c/Users/jacwe/Downl

In [3]:
import anndata as ad
import scanpy as sc

if RNA_H5AD.exists():
    print(f"[RNA] Found existing {RNA_H5AD}, loading...")
    rna = ad.read_h5ad(RNA_H5AD)
else:
    raise FileNotFoundError(
        f"RNA h5ad not found: {RNA_H5AD}\n"
        "Please provide rna_scanpy_preprocessed.h5ad in the input directory."
    )

print("\n[RNA] Summary:")
print(rna)
print("Number of cells:", rna.n_obs)
print("Number of genes:", rna.n_vars)

assert rna.n_obs > 0 and rna.n_vars > 0, "Empty RNA object?"

rna.var_names_make_unique()
rna.obs_names_make_unique()

[RNA] Found existing /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/rna_scanpy_preprocessed.h5ad, loading...

[RNA] Summary:
AnnData object with n_obs × n_vars = 33054 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'RNA_snn_res.0.5', 'seurat_clusters', 'predicted_celltype', 'celltype', 'modality', 'nCount_ATAC', 'nFeature_ATAC', 'ATAC_snn_res.0.1', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_ACTIVITY', 'nFeature_ACTIVITY', 'predicted.id', 'prediction.score.GABAergic.ACs', 'prediction.score.ON.BPs', 'prediction.score.rod.BPs', 'prediction.score.resting.MG', 'prediction.score.RGCs', 'prediction.score.OFF.BPs', 'prediction.score.cone.PRs', 'prediction.score.Glycinergic.ACs', 'prediction.score.BPs', 'prediction.score.oligos', 'prediction.score.microglia', 'prediction.score.HCs', 'prediction.score.NIRG.cells', 'prediction.score.rod.PRs', 'prediction.score.activated.MG', 'prediction.score.max', 'n_genes_by_counts', 'lo

In [4]:
import pickle, re, os
import warnings
import pandas as pd
import pyranges as pr
from pathlib import Path
from pycisTopic.cistopic_class import CistopicObject
from pycisTopic.topic_binarization import binarize_topics

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"pyranges\.out",
    message=r"Passing a set as an indexer is deprecated.*"
)


def load_cistopic(chrstr_pkl: Path) -> CistopicObject:
    if not chrstr_pkl.exists():
        raise FileNotFoundError(f"cisTopic pickle not found: {chrstr_pkl}")
    print(f"[ATAC] Loading cisTopic object: {chrstr_pkl}")
    with open(chrstr_pkl, "rb") as fh:
        return pickle.load(fh)


def _regions_to_pyranges(idx_like, label="") -> pr.PyRanges:
    """
    Convert region IDs of form ':1:100-200' or '1:100-200' to PyRanges.
    Chromosomes kept as bare numbers (no chr prefix) to match DB format.
    """
    idx_series = pd.Index(idx_like).to_series()
    pat = r"^:?(?P<chrom>[^:]+):(?P<start>\d+)-(?P<end>\d+)$"
    parsed = idx_series.str.extract(pat)

    if parsed.isna().any().any():
        print(f"[REGIONSETS] ERROR parsing region IDs for {label}, bad examples:")
        print(idx_series[parsed.isna().any(axis=1)].head())
        raise RuntimeError(f"Region ID pattern mismatch for {label}")

    # Keep chromosomes as bare numbers — do NOT add chr prefix
    chrom = parsed["chrom"].astype(str)
    start = parsed["start"].astype(int)
    end   = parsed["end"].astype(int)

    df = pd.DataFrame({"Chromosome": chrom, "Start": start, "End": end})
    return pr.PyRanges(df)


def _safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]", "_", str(name))


cistopic = load_cistopic(CISTOPIC_PKL)

print("\n[ATAC] cisTopic object summary:")
print("  #cells:   ", len(getattr(cistopic, "cell_names", [])))
print("  #regions: ", len(getattr(cistopic, "region_names", [])))

# ------------------------------------------------------------------
# DEM region sets: topic binarization + export BEDs
# ------------------------------------------------------------------
if REGION_SETS_PKL.exists():
    print(f"\n[REGIONSETS] Loading existing topic-binarized region sets: {REGION_SETS_PKL}")
    with open(REGION_SETS_PKL, "rb") as fh:
        region_sets = pickle.load(fh)
else:
    print("\n[REGIONSETS] Running binarize_topics (Otsu)...")
    region_sets = binarize_topics(
        cistopic_obj=cistopic,
        target="region",
        method="otsu",
        smooth_topics=True,
        plot=False
    )
    with open(REGION_SETS_PKL, "wb") as fh:
        pickle.dump(region_sets, fh)
    print(f"[REGIONSETS] Wrote topic-binarized region sets: {REGION_SETS_PKL}")

REGION_SETS_DIR.mkdir(exist_ok=True)
topics_dir = REGION_SETS_DIR / "Topics"
topics_dir.mkdir(exist_ok=True)

print("\n[REGIONSETS] Exporting region sets to BED files...")

if isinstance(region_sets, dict):
    for topic_name, topic_regions in region_sets.items():
        label = str(topic_name)
        pr_obj = _regions_to_pyranges(topic_regions.index, label=label)
        bed_name = _safe_name(topic_name) + ".bed"
        bed_path = topics_dir / bed_name
        pr_obj.to_bed(str(bed_path))
        print("  wrote:", bed_path)
else:
    pr_obj = _regions_to_pyranges(region_sets.index, label="root")
    bed_path = topics_dir / "regions.bed"
    pr_obj.to_bed(str(bed_path))
    print("  wrote:", bed_path)

print("\n[REGIONSETS] Done. Use this as `region_set_folder` in config.yaml:")
print("   ", topics_dir)

# Sanity: DEM region sets must be subset of cistopic regions
cisto_region_set = set(cistopic.region_names)
demo_rs_any = set()
if isinstance(region_sets, dict):
    for _, rs in region_sets.items():
        demo_rs_any |= set(rs.index.astype(str))
else:
    demo_rs_any = set(region_sets.index.astype(str))

overlap = len(demo_rs_any & cisto_region_set)
print(f"\n[REGIONSETS] DEM region IDs: {len(demo_rs_any):,}")
print("[REGIONSETS] Overlap with cisTopic regions:", overlap)
if overlap < 0.8 * len(demo_rs_any):
    raise RuntimeError(
        "DEM region sets poorly overlap cisTopic regions – check region ID formatting."
    )

# Quick BED check — verify chromosomes are bare numbers
sample_bed = sorted(topics_dir.glob("Topic*.bed"))[0]
sample_df = pd.read_csv(sample_bed, sep="\t", header=None, names=["Chromosome","Start","End"])
sample_chroms = sorted(sample_df["Chromosome"].astype(str).unique())[:5]
print(f"\n[REGIONSETS] Sample BED chromosomes (should be bare numbers like '1','2'): {sample_chroms}")
if any(c.lower().startswith("chr") for c in sample_chroms):
    raise RuntimeError(
        "BED files have 'chr' prefix but DB uses bare numbers! "
        "Check _regions_to_pyranges — chr prefix should not be added."
    )

[ATAC] Loading cisTopic object: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/atac_cistopic_selected_model_chrstr.pkl

[ATAC] cisTopic object summary:
  #cells:    15000
  #regions:  50187

[REGIONSETS] Loading existing topic-binarized region sets: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets_topics_otsu.pkl

[REGIONSETS] Exporting region sets to BED files...
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic1.bed
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic2.bed
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic3.bed
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic4.bed
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic5.bed
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic6.bed
  wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region

In [5]:
import pandas as pd
from pathlib import Path

BASE = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input")

# Load the motif order used to build the DB
lst_path = BASE / "motifs_chicken_expr_for_db.lst"
motif_order = [line.strip() for line in lst_path.read_text().splitlines() if line.strip()]
db_motif_ids = set(motif_order)
print(f"DB motif IDs: {len(db_motif_ids)}")

# Load raw annotation
raw_tbl = BASE / "v10nr_clust_public" / "snapshots" / "motifs-v10-nr.chicken-m0.00001-o0.0.tbl"
raw_mdf = pd.read_csv(raw_tbl, sep="\t", low_memory=False)
print(f"Raw tbl shape: {raw_mdf.shape}")
print(f"Columns: {list(raw_mdf.columns)}")

# Filter to only rows whose #motif_id is in the DB
raw_mdf["#motif_id"] = raw_mdf["#motif_id"].astype(str)
df_db = raw_mdf[raw_mdf["#motif_id"].isin(db_motif_ids)].copy()
print(f"\nAfter filtering to DB motifs: {df_db.shape}")
print(f"Unique #motif_id: {df_db['#motif_id'].nunique()}")  # should be 1639

# Drop rows with no TF annotation
tf_col = "gene_name"
df_db = df_db[df_db[tf_col].notna() & (df_db[tf_col].astype(str).str.strip() != "")].copy()
print(f"After dropping null TF: {df_db.shape}")
print(f"Unique motifs with TF: {df_db['#motif_id'].nunique()}")

# How many DB motifs have NO TF annotation at all?
motifs_with_tf = set(df_db["#motif_id"].astype(str))
motifs_without_tf = db_motif_ids - motifs_with_tf
print(f"DB motifs with no TF annotation: {len(motifs_without_tf)}")
print(f"Sample unannotated: {list(motifs_without_tf)[:5]}")

# Rename to what SCENIC+ expects
df_db = df_db.rename(columns={"#motif_id": "#motif_id"})  # no rename
df_db["TF"] = df_db["gene_name"]  # add alias column
df_db.to_csv(INDIR / "motifs.filtered.tbl", sep="\t", index=False)

# Save — do NOT enforce 1-TF-per-motif, clusters legitimately map to multiple TFs
MOTIF_ANNOT_FILT = BASE / "motifs.filtered.tbl"
MOTIF_ANNOT_FILT.unlink(missing_ok=True)
df_db.to_csv(MOTIF_ANNOT_FILT, sep="\t", index=False)
print(f"\nWrote: {MOTIF_ANNOT_FILT}")
print(f"Shape: {df_db.shape}")
print(f"Unique TFs: {df_db['TF'].nunique()}")
print(f"\nSample rows:")
print(df_db[["#motif_id","TF"]].drop_duplicates().head(10).to_string())

import subprocess
result = subprocess.run(["python", "-c", """
from pycistarget.motif_enrichment_cistarget import cisTarget
import inspect
src = inspect.getsource(cisTarget._add_motif_annotation)
# Find where TF is assigned
for i, line in enumerate(src.split('\\n')):
    if 'TF' in line or 'gene_name' in line or 'annot' in line.lower():
        print(f'{i}: {line}')
"""], capture_output=True, text=True)
print(result.stdout[:2000])
print(result.stderr[-500:] if result.stderr else "")

DB motif IDs: 1639
Raw tbl shape: (186920, 13)
Columns: ['#motif_id', 'motif_name', 'motif_description', 'source_name', 'source_version', 'gene_name', 'motif_similarity_qvalue', 'similar_motif_id', 'similar_motif_description', 'orthologous_identity', 'orthologous_gene_name', 'orthologous_species', 'description']

After filtering to DB motifs: (84973, 13)
Unique #motif_id: 1639
After dropping null TF: (84973, 13)
Unique motifs with TF: 1639
DB motifs with no TF annotation: 0
Sample unannotated: []

Wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/motifs.filtered.tbl
Shape: (84973, 14)
Unique TFs: 662

Sample rows:
           #motif_id        TF
0  metacluster_196.3     HNF4A
1  metacluster_196.3     HNF4G
2  metacluster_196.3  HNF4beta
3  metacluster_196.3     NR1D1
4  metacluster_196.3     NR1D2
5  metacluster_196.3     NR1H3
6  metacluster_196.3     NR1H4
7  metacluster_196.3     NR2C1
8  metacluster_196.3     NR2C2
9  metacluster_196.3     NR2F1

  from pkg_resources import

In [6]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as pf
from pathlib import Path

BASE = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input")
lst_path = BASE / "motifs_chicken_expr_for_db.lst"
motif_order = [l.strip() for l in lst_path.read_text().splitlines() if l.strip()]

for fname in [
    "gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather",
    "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather",
]:
    bak = BASE / (fname + ".bak")
    fpath = BASE / fname
    
    if bak.exists():
        print(f"Restoring from backup: {bak.name}")
        df = pd.read_feather(bak)
    else:
        print(f"No backup found, reading current: {fpath.name}")
        # Try polars as fallback since pyarrow is failing
        import polars as pl
        df = pl.read_ipc(str(fpath)).to_pandas()
    
    if "motifs" in df.columns:
        df = df.drop(columns=["motifs"])
    df.insert(0, "motifs", motif_order)
    
    # Write as uncompressed pyarrow feather v2
    table = pa.Table.from_pandas(df, preserve_index=False)
    pf.write_feather(table, fpath, compression="uncompressed")
    print(f"  Wrote {fpath.name}: shape={df.shape}, cols[0]={df.columns[0]}")

Restoring from backup: gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather.bak
  Wrote gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather: shape=(1639, 117522), cols[0]=motifs
Restoring from backup: gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather.bak
  Wrote gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather: shape=(1639, 117522), cols[0]=motifs


In [7]:
# Restore motif IDs as index in both feather files
for fname in [
    "gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather",
    "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather",
]:
    fpath = BASE / fname
    df = pd.read_feather(fpath)
    
    # Drop the 'motifs' column if a previous partial fix already added it
    if "motifs" in df.columns:
        df = df.drop(columns=["motifs"])
    
    assert len(motif_order) == len(df), \
        f"Length mismatch: list={len(motif_order)}, DB={len(df)}"
    
    df.insert(0, "motifs", motif_order)
    import pyarrow as pa
    import pyarrow.feather as pf
    table = pa.Table.from_pandas(df, preserve_index=False)
    pf.write_feather(table, fpath, compression="uncompressed")
    
    # Verify round-trip
    df_check = pd.read_feather(fpath)
    print(f"\n{fname}")
    print(f"  columns[0]: {df_check.columns[0]}")
    print(f"  sample motif IDs: {df_check['motifs'].iloc[:3].tolist()}")
    print(f"  shape: {df_check.shape}")


gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather
  columns[0]: motifs
  sample motif IDs: ['metacluster_196.3', 'bergman__bcd', 'metacluster_149.1']
  shape: (1639, 117522)

gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
  columns[0]: motifs
  sample motif IDs: ['metacluster_196.3', 'bergman__bcd', 'metacluster_149.1']
  shape: (1639, 117522)


In [8]:
# ------------------------------------------------------------------
# cisTarget DBs: sanity-check & expose region IDs for compatibility
# ------------------------------------------------------------------

for p in (CTX_DB_RANKINGS, CTX_DB_SCORES):
    if not p.exists():
        raise FileNotFoundError(f"cisTarget DB feather missing: {p}")

print("[CTX] Loading scores DB:", CTX_DB_SCORES)
scores_df = pd.read_feather(CTX_DB_SCORES)
print("[CTX] Scores shape:", scores_df.shape)

print("[CTX] Loading rankings DB:", CTX_DB_RANKINGS)
rankings_df = pd.read_feather(CTX_DB_RANKINGS)
print("[CTX] Rankings shape:", rankings_df.shape)

print("[CTX] Scores columns (first 10):", scores_df.columns[:10].tolist())
print("[CTX] Rankings columns (first 10):", rankings_df.columns[:10].tolist())

# In these regions_vs_motifs databases:
#   - ROWS   = motifs
#   - COLUMNS = regions (strings like "::10:1000509-1001366")
# There is no explicit motif ID column; motif IDs live in the row index.
# The 1–1 TF–motif mapping is fully handled by the motif annotation table,
# which we already filtered in Cell 4.

# Basic structural checks
if list(scores_df.columns) != list(rankings_df.columns):
    print("[CTX] WARNING: scores and rankings have different region columns.")
else:
    print("[CTX] Region columns match between scores and rankings (OK).")

num_db_motifs_scores   = scores_df.shape[0]
num_db_motifs_rankings = rankings_df.shape[0]
if num_db_motifs_scores != num_db_motifs_rankings:
    print("[CTX] WARNING: scores and rankings have different # rows (motifs).")
print(f"[CTX] #motifs (rows) in scores   : {num_db_motifs_scores}")
print(f"[CTX] #motifs (rows) in rankings : {num_db_motifs_rankings}")

# Compare motif counts — read from saved file since filtered_motif_ids may not be in scope
_filt_tbl = pd.read_csv(MOTIF_ANNOT_FILT, sep="\t")
_filt_motif_id_col = next((c for c in ["MotifID","#motif_id","motif_id"] if c in _filt_tbl.columns), None)
_n_filt = _filt_tbl[_filt_motif_id_col].nunique() if _filt_motif_id_col else "?"

# ------------------------------------------------------------------
# Region IDs from DB for compatibility checks (with cisTopic)
# ------------------------------------------------------------------

def _normalize_region_id_db(s: str) -> str:
    """
    Normalize region IDs from DB so they can be compared with cisTopic region_names.
    DB columns look like '::10:1000509-1001366'.
    Our cisTopic IDs look like ':10:1000509-1001366'.
    We'll:
      - ensure string
      - collapse leading '::' to ':'
    """
    s = str(s)
    if s.startswith("::"):
        return s[1:]  # '::10:...' -> ':10:...'
    return s

region_cols_db = list(scores_df.columns)
print(f"[CTX] #region columns in DB: {len(region_cols_db)}")

ctx_region_ids = { _normalize_region_id_db(c) for c in region_cols_db }
print(f"[CTX] #normalized DB region IDs (for compatibility checks): {len(ctx_region_ids)}")

print("[CTX] cisTarget DB sanity-check completed. No modifications were made to the feather files.")


[CTX] Loading scores DB: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather
[CTX] Scores shape: (1639, 117522)
[CTX] Loading rankings DB: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
[CTX] Rankings shape: (1639, 117522)
[CTX] Scores columns (first 10): ['motifs', '::10:1000509-1001366', '::10:10005520-10006430', '::10:10009066-10009865', '::10:1001544-1002266', '::10:10017106-10017908', '::10:10018032-10018630', '::10:10028063-10028820', '::10:10029333-10030212', '::10:10030854-10031657']
[CTX] Rankings columns (first 10): ['motifs', '::10:1000509-1001366', '::10:10005520-10006430', '::10:10009066-10009865', '::10:1001544-1002266', '::10:10017106-10017908', '::10:10018032-10018630', '::10:10028063-10028820', '::10:10029333-10030212', '::10:10030854-10031657']
[CTX] Region columns match between scores and rankings (OK).
[CTX] #motifs (rows) in scores   

In [10]:
import pandas as pd
import pyranges as pr
import pycistarget.utils as pctu
from pathlib import Path

BASE = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input")
topics_dir = BASE / "region_sets" / "Topics"

bed_path = sorted(topics_dir.glob("Topic1.bed"))[0]
print("[CHECK] Using BED:", bed_path)

bed_df = pd.read_csv(
    bed_path,
    sep="\t",
    header=None,
    usecols=[0, 1, 2],
    names=["Chromosome", "Start", "End"],
    dtype={"Chromosome": str, "Start": int, "End": int}
)

print("[CHECK] BED head (post-patch, direct from disk):")
print(bed_df.head())

pr_bed = pr.PyRanges(bed_df)

# DB side
db_path = BASE / "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather"
print("[CHECK] Loading DB from:", db_path)
db = pd.read_feather(db_path)
db_regions = list(db.columns)
db_coords = pctu.region_names_to_coordinates(db_regions)
print("[CHECK] DB coords head:")
print(db_coords.head())

pr_db = pr.PyRanges(db_coords)

print("[CHECK] Unique chromosomes in BED:", sorted(pr_bed.Chromosome.unique())[:20])
print("[CHECK] Unique chromosomes in DB:", sorted(pr_db.Chromosome.unique())[:20])

overlap = pr_bed.join(pr_db)
print("[CHECK] # overlaps between BED and DB (no extra normalization):", overlap.df.shape[0])


[CHECK] Using BED: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/region_sets/Topics/Topic1.bed
[CHECK] BED head (post-patch, direct from disk):
  Chromosome      Start        End
0          1  121949495  121950440
1          1  185126885  185127823
2          1   14406796   14407694
3          1  188941837  188942730
4          1   56076714   56077640
[CHECK] Loading DB from: /mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
[CHECK] DB coords head:
  Chromosome     Start       End
0         10   1000509   1001366
1         10  10005520  10006430
2         10  10009066  10009865
3         10   1001544   1002266
4         10  10017106  10017908
[CHECK] Unique chromosomes in BED: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27']
[CHECK] Unique chromosomes in DB: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22',

In [11]:
from pathlib import Path
import pandas as pd

BASE = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input")

db_files = [
    BASE / "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather",
    BASE / "gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather",
    BASE / "gallus_consensus_chicken_expr.motifs_vs_regions.scores.feather",
]

def normalize_region_name(name: str) -> str:
    s = str(name).strip()
    if not s:
        return s

    # Drop anything before the last '::' (if present)
    if "::" in s:
        s = s.split("::")[-1]

    # Drop leading colons
    while s.startswith(":"):
        s = s[1:]

    # Strip leading 'chr' if present
    if s.startswith("chr"):
        s = s[3:]

    return s

for fpath in db_files:
    if not fpath.exists():
        print(f"[DB PATCH] Skipping (not found): {fpath}")
        continue

    print(f"[DB PATCH] Normalizing columns in: {fpath.name}")
    df = pd.read_feather(fpath)

    old_cols = list(df.columns)
    new_cols = []
    changed = 0

    for c in old_cols:
        # Only touch columns that *look* like genomic coordinates
        if isinstance(c, str) and ":" in c and "-" in c:
            nc = normalize_region_name(c)
            if nc != c:
                changed += 1
            new_cols.append(nc)
        else:
            new_cols.append(c)

    df.columns = new_cols
    print(f"  - Columns changed: {changed} / {len(old_cols)}")

    # Backup original once, then overwrite
    backup = fpath.with_suffix(fpath.suffix + ".bak")
    if not backup.exists():
        df_orig = pd.read_feather(fpath)
        df_orig.to_feather(backup)
        print(f"  - Backed up original to: {backup.name}")

    import pyarrow as pa
    import pyarrow.feather as pf
    table = pa.Table.from_pandas(df, preserve_index=False)
    pf.write_feather(table, fpath, compression="uncompressed")
    print(f"  - Wrote normalized DB to: {fpath.name}")

print("[DB PATCH] Done.")

BASE = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input")
rank_f = BASE / "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather"

df = pd.read_feather(rank_f)
cols = [c for c in df.columns if isinstance(c, str) and ":" in c and "-" in c]
print("[CHECK] Example normalized region columns:")
print(cols[:10])

[DB PATCH] Normalizing columns in: gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
  - Columns changed: 117521 / 117522
  - Wrote normalized DB to: gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
[DB PATCH] Normalizing columns in: gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather
  - Columns changed: 117521 / 117522
  - Wrote normalized DB to: gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather
[DB PATCH] Normalizing columns in: gallus_consensus_chicken_expr.motifs_vs_regions.scores.feather
  - Columns changed: 0 / 1640
  - Wrote normalized DB to: gallus_consensus_chicken_expr.motifs_vs_regions.scores.feather
[DB PATCH] Done.
[CHECK] Example normalized region columns:
['10:1000509-1001366', '10:10005520-10006430', '10:10009066-10009865', '10:1001544-1002266', '10:10017106-10017908', '10:10018032-10018630', '10:10028063-10028820', '10:10029333-10030212', '10:10030854-10031657', '10:1003341-1004209']


In [12]:
import gc
# Clear large dataframes loaded by cell 13
for var in ['df', 'table', 'scores_df', 'rankings_df']:
    if var in dir():
        del globals()[var]
gc.collect()
print("Memory cleared")

Memory cleared


In [14]:
# ================================================================
# SANITY CHECK: Verify DEM inputs before running Snakemake
# ================================================================
import pandas as pd
import pyranges as pr
import pycistarget.utils as pctu
from pathlib import Path

BASE = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/scenicplus_input")
topics_dir = BASE / "region_sets" / "Topics"
scores_db_path = BASE / "gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather"

print("=" * 60)
print("SANITY CHECK: DEM inputs")
print("=" * 60)

# --- 1. Check BED file column layout ---
print("\n[1] BED file column layout check:")
for bed_path in sorted(topics_dir.glob("Topic*.bed"))[:3]:
    raw = bed_path.read_text().split("\n")
    print(f"  {bed_path.name} first line: {repr(raw[0])}")
    df = pd.read_csv(bed_path, sep="\t", header=None)
    print(f"  {bed_path.name} shape: {df.shape}, cols[0] sample: {df.iloc[:3, 0].tolist()}")

# --- 2. Check chromosomes in BED are bare numbers ---
print("\n[2] Chromosome format in BED files:")
all_ok = True
for bed_path in sorted(topics_dir.glob("Topic*.bed")):
    df = pd.read_csv(bed_path, sep="\t", header=None, names=["Chromosome","Start","End"],
                     usecols=[0,1,2])
    chroms = df["Chromosome"].astype(str).unique()
    bad = [c for c in chroms if c == "." or c.lower().startswith("chr") or not c[0].isdigit()]
    if bad:
        print(f"  PROBLEM {bed_path.name}: bad chroms = {bad[:5]}")
        all_ok = False
    else:
        pass  # quiet if OK
if all_ok:
    print("  All BED files have correct bare-number chromosomes.")

# --- 3. Check overlap between Topic1 BED and DB ---
print("\n[3] BED vs DB overlap (Topic1):")
bed_df = pd.read_csv(
    topics_dir / "Topic1.bed", sep="\t", header=None,
    names=["Chromosome","Start","End"], usecols=[0,1,2]
)
bed_df["Chromosome"] = bed_df["Chromosome"].astype(str)
bed_df["Start"] = pd.to_numeric(bed_df["Start"], errors="coerce")
bed_df["End"] = pd.to_numeric(bed_df["End"], errors="coerce")
bed_df = bed_df.dropna()
print(f"  BED rows: {len(bed_df)}, chroms: {sorted(bed_df['Chromosome'].unique())[:5]}")

db = pd.read_feather(scores_db_path)
region_cols = [c for c in db.columns if isinstance(c, str) and ":" in c and "-" in c]
db_coords = pctu.region_names_to_coordinates(region_cols)
print(f"  DB regions: {len(db_coords)}, chroms: {sorted(db_coords['Chromosome'].unique())[:5]}")

pr_bed = pr.PyRanges(bed_df)
pr_db = pr.PyRanges(db_coords)
overlap = pr_bed.join(pr_db)
n_overlap = overlap.df.shape[0]
print(f"  Overlap: {n_overlap} regions")
if n_overlap == 0:
    print("  !! CRITICAL: Zero overlap — DEM will produce empty results !!")
elif n_overlap < 100:
    print(f"  !! WARNING: Very low overlap ({n_overlap}) — DEM may produce sparse results")
else:
    print(f"  OK: Good overlap ({n_overlap} regions)")

# --- 4. Check fraction_overlap threshold ---
# DEM requires fraction_overlap_w_dem_database of foreground regions to hit DB
# Default is 0.4 — check how many foreground regions actually overlap
n_bed = len(bed_df)
frac = n_overlap / n_bed if n_bed > 0 else 0
print(f"\n[4] Foreground overlap fraction: {n_overlap}/{n_bed} = {frac:.3f}")
print(f"    Config threshold: 0.4")
if frac < 0.4:
    print(f"    !! PROBLEM: Only {frac:.1%} of foreground regions overlap DB.")
    print(f"       DEM will skip this topic (needs >= 40%).")
    print(f"       Consider lowering fraction_overlap_w_dem_database in config.yaml")
else:
    print(f"    OK: Fraction {frac:.1%} meets the 0.4 threshold.")

# --- 5. Check motif annotation file ---
print("\n[5] Motif annotation file:")
motif_tbl = BASE / "motifs.filtered.tbl"
if not motif_tbl.exists():
    print("  !! MISSING: motifs.filtered.tbl not found!")
else:
    mdf = pd.read_csv(motif_tbl, sep="\t", nrows=5)
    print(f"  Columns: {list(mdf.columns)}")
    # pycistarget expects '#motif_id' or 'MotifID' column
    has_motif_id = any(c in mdf.columns for c in ["#motif_id", "MotifID", "motif_id"])
    has_tf = any(c in mdf.columns for c in ["gene_name", "TF", "tf"])
    print(f"  Has motif ID column: {has_motif_id}")
    print(f"  Has TF column: {has_tf}")
    if not has_motif_id or not has_tf:
        print("  !! PROBLEM: motifs.filtered.tbl is missing expected columns!")
        print(f"     pycistarget needs '#motif_id'/'MotifID' and 'gene_name'/'TF'")

# --- 6. Check DB motif index vs motif annotation intersection ---
print("\n[6] DB motif index vs motif annotation intersection:")
db_motif_index = list(db.index)
print(f"  DB motif index sample: {db_motif_index[:3]}")
full_mdf = pd.read_csv(motif_tbl, sep="\t")
motif_id_col = next((c for c in ["MotifID","#motif_id","motif_id"] if c in full_mdf.columns), None)
if motif_id_col:
    annot_motifs = set(full_mdf[motif_id_col].astype(str))
    db_motifs = set(str(m) for m in db_motif_index)
    intersection = annot_motifs & db_motifs
    print(f"  Annotation motifs: {len(annot_motifs)}")
    print(f"  DB motifs: {len(db_motifs)}")
    print(f"  Intersection: {len(intersection)}")
    if len(intersection) == 0:
        print("  !! CRITICAL: No motif ID overlap between annotation and DB!")
        print("     Sample annotation IDs:", list(annot_motifs)[:3])
        print("     Sample DB IDs:", list(db_motifs)[:3])
    elif len(intersection) < 100:
        print(f"  !! WARNING: Very few matching motifs ({len(intersection)})")
    else:
        print(f"  OK: {len(intersection)} motifs in common.")

print("\n" + "=" * 60)
print("SANITY CHECK COMPLETE")
print("=" * 60)

SANITY CHECK: DEM inputs

[1] BED file column layout check:
  Topic1.bed first line: '1\t121949495\t121950440\t.\t.\t.'
  Topic1.bed shape: (4851, 6), cols[0] sample: ['1', '1', '1']
  Topic10.bed first line: '1\t119827559\t119828515\t.\t.\t.'
  Topic10.bed shape: (4309, 6), cols[0] sample: ['1', '1', '1']
  Topic11.bed first line: '1\t176425340\t176426215\t.\t.\t.'
  Topic11.bed shape: (5101, 6), cols[0] sample: ['1', '1', '1']

[2] Chromosome format in BED files:
  PROBLEM Topic1.bed: bad chroms = ['JAENSK010000034.1', 'JAENSK010000477.1', 'JAENSK010000484.1', 'JAENSK010000584.1', 'JAENSK010000592.1']
  PROBLEM Topic10.bed: bad chroms = ['JAENSK010000483.1', 'JAENSK010000588.1', 'JAENSK010000635.1', 'MU179261.1', 'MU179271.1']
  PROBLEM Topic11.bed: bad chroms = ['JAENSK010000296.1', 'JAENSK010000448.1', 'JAENSK010000478.1', 'JAENSK010000479.1', 'JAENSK010000483.1']
  PROBLEM Topic12.bed: bad chroms = ['JAENSK010000452.1', 'JAENSK010000477.1', 'JAENSK010000479.1', 'JAENSK010000483.1'

In [16]:
import mudata as mu

OUT = "/mnt/c/Users/jacwe/Downloads/SAMAP/scplus_pipeline/Snakemake/ACC_GEX.fixed3.h5mu"
m = mu.read_h5mu(OUT)

atac = m.mod["scATAC"]
print("Before:", atac.var_names[:5].tolist())

# Strip leading colon from region names
atac.var_names = atac.var_names.str.lstrip(":")
print("After:", atac.var_names[:5].tolist())

# Update coordinates to match
import pandas as pd
def parse_region(name):
    parts = name.split(":")
    chrom = parts[0]
    start, end = parts[1].split("-")
    return chrom, int(start), int(end)

chroms, starts, ends = [], [], []
for name in atac.var_names:
    c, s, e = parse_region(name)
    chroms.append(c)
    starts.append(s)
    ends.append(e)

atac.var["Chromosome"] = chroms
atac.var["Start"] = starts
atac.var["End"] = ends

m.update()
m.write_h5mu(OUT)
print(f"Wrote: {OUT}")
print("Sample var_names:", atac.var_names[:5].tolist())

Before: ['1:33501-34428', '1:38097-38997', '1:50233-51020', '1:55018-55937', '1:60837-61594']
After: ['1:33501-34428', '1:38097-38997', '1:50233-51020', '1:55018-55937', '1:60837-61594']
Wrote: /mnt/c/Users/jacwe/Downloads/SAMAP/scplus_pipeline/Snakemake/ACC_GEX.fixed3.h5mu
Sample var_names: ['1:33501-34428', '1:38097-38997', '1:50233-51020', '1:55018-55937', '1:60837-61594']


# SCENIC+ Package Patches Applied
This notebook required the following patches to installed SCENIC+ packages
due to incompatibilities with newer versions of pyranges, pandas, and pyarrow.

## Files Modified

### 1. `pycistarget/utils.py`
**Function:** `region_names_to_coordinates`  
**Problem:** Didn't handle Gallus gallus region name format (`:1:start-end`)  
**Fix:** Added parsing for `:chrom:start-end`, `::chrom:start-end`, and `chr:start-end` formats

### 2. `scenicplus/cli/commands.py`
**Functions:** `prepare_GEX_ACC`, `infer_grn`, `calculate_auc`, `create_scplus_mudata`  
**Problems:**
- scATAC.var Categorical columns caused h5mu write failure
- Empty direct eRegulon (expected for non-model organisms) caused unhandled crashes
**Fix:** Sanitize Categorical columns; gracefully handle empty direct eRegulon at every downstream step

### 3. `scenicplus/data_wrangling/cistarget_wrangling.py`
**Problem:** `set.union()` called on empty cistrome list  
**Fix:** `set.union(*sets) if sets else []`

### 4. `scenicplus/utils.py`
**Problems:**
- `_pd.DataFrame` undefined reference
- `extend_pyranges` failed on empty strand subsets
- `reduce_pyranges_b` assumed Strand attribute always present
- `calculate_distance_join` used PyRanges attribute access incompatible with newer pyranges
**Fix:** All four issues patched; `calculate_distance_join` fully rewritten in pure pandas

### 5. `scenicplus/data_wrangling/gene_search_space.py`
**Problem:** `get_search_space` used `pr.join()` which silently returns empty results in newer pyranges; also multiple `.df` attribute access failures  
**Fix:** Entire `get_search_space` function replaced with pure-pandas implementation using chromosome-by-chromosome interval overlap; memory-efficient using `intervaltree`

### 6. `ctxcore/ctdb.py`
**Problems:** `pl.read_ipc_schema(file=...)` keyword arg; `if not index_column_idx` logic error  
**Fix:** Positional arg; `if index_column_idx is None`

### 7. `ACC_GEX.fixed3.h5mu` (user data file)
**Problem:** scATAC region names had leading colon (`:1:start-end`) incompatible with search_space output format  
**Fix:** Stripped leading colon from all ATAC var_names; recomputed Chromosome/Start/End coordinates

In [17]:
import subprocess, os
from pathlib import Path

os.chdir("/mnt/c/Users/jacwe/Downloads/SAMAP/scplus_pipeline/Snakemake")
subprocess.run(["snakemake", "--unlock"], capture_output=False)

# Only clear outputs that need to be re-run (keep tf_to_gene, region_to_gene, search_space)
for f in ["dem_results.hdf5", "dem_results.html",
          "ctx_results.hdf5", "ctx_results.html",
          "tf_names.txt", "cistromes_direct.h5ad", "cistromes_extended.h5ad",
          "eRegulon_direct.tsv", "eRegulons_extended.tsv",
          "AUCell_direct.h5mu", "AUCell_extended.h5mu",
          "scplusmdata.h5mu"]:
    Path(f).unlink(missing_ok=True)

result = subprocess.run(
    ["snakemake", "--cores", "8", "--rerun-incomplete", "--latency-wait", "500"],
    capture_output=False, text=True,
    env={**os.environ, "PYTHONWARNINGS": "ignore"}
)

Building DAG of jobs...
Unlocked working directory.
Assuming unrestricted shared filesystem usage for local execution.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 8
Rules claiming more threads will be scaled down.
Job stats:
job                           count
--------------------------  -------
AUCell_direct                     1
AUCell_extended                   1
all                               1
eGRN_direct                       1
eGRN_extended                     1
get_search_space                  1
motif_enrichment_cistarget        1
motif_enrichment_dem              1
prepare_menr                      1
region_to_gene                    1
scplus_mudata                     1
tf_to_gene                        1
total                            12

Select jobs to execute...
Execute 1 jobs...

[Sun Mar 29 16:39:18 2026]
localrule motif_enrichment_dem:
    input: ../../scenicplus_input/region_sets, ../../scenicplus_input/gallus_consensus_chicken_expr.regions

2026-03-29 16:39:24,792 SCENIC+      INFO     Reading region sets from: ../../scenicplus_input/region_sets
2026-03-29 16:39:24,799 SCENIC+      INFO     Reading all .bed files in: Topics
2026-03-29 16:39:31,711 DEM          INFO     Running DEM for Topics_Topic2_vs_all
2026-03-29 16:39:31,715 DEM          INFO     Running DEM for Topics_Topic11_vs_all
2026-03-29 16:39:31,759 DEM          INFO     Running DEM for Topics_Topic10_vs_all
2026-03-29 16:39:31,786 DEM          INFO     Running DEM for Topics_Topic12_vs_all
2026-03-29 16:39:31,787 DEM          INFO     Running DEM for Topics_Topic1_vs_all
2026-03-29 16:39:31,792 DEM          INFO     Running DEM for Topics_Topic3_vs_all
2026-03-29 16:39:31,797 DEM          INFO     Running DEM for Topics_Topic4_vs_all
2026-03-29 16:39:31,806 DEM          INFO     Running DEM for Topics_Topic5_vs_all
2026-03-29 16:39:41,943 DEM          INFO     Adding motif-to-TF annotation
2026-03-29 16:39:42,317 DEM          INFO     Adding motif-to-TF annot

[Sun Mar 29 16:39:49 2026]
Finished job 7.
1 of 12 steps (8%) done
Select jobs to execute...
Execute 1 jobs...

[Sun Mar 29 16:39:49 2026]
localrule motif_enrichment_cistarget:
    input: ../../scenicplus_input/region_sets, ../../scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather, ../../scenicplus_input/motifs.filtered.tbl
    output: ctx_results.hdf5, ctx_results.html
    jobid: 9
    reason: Missing output files: ctx_results.hdf5; Updated input files: ../../scenicplus_input/motifs.filtered.tbl, ../../scenicplus_input/gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather
    threads: 8
    resources: tmpdir=/tmp



2026-03-29 16:39:53,062 SCENIC+      INFO     Reading region sets from: ../../scenicplus_input/region_sets
2026-03-29 16:39:53,065 SCENIC+      INFO     Reading all .bed files in: Topics
2026-03-29 16:39:57,197 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,199 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,201 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,273 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,273 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,274 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,278 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:39:57,292 cisTarget    INFO     Reading cisTarget database
2026-03-29 16:40:02,859 cisTarget    INFO     Running cisTarget for Topics_Topic2 which has 3460 regions
2026-03-29 16:40:03,202 cisTarget    INFO     Running cisTarget for Topics_Topic10 which has 4254 regions
2026-03-29 16:40:0

[Sun Mar 29 16:40:11 2026]
Finished job 9.
2 of 12 steps (17%) done
Select jobs to execute...
Execute 2 jobs...

[Sun Mar 29 16:40:11 2026]
localrule prepare_menr:
    input: dem_results.hdf5, ctx_results.hdf5, ACC_GEX.fixed3.h5mu
    output: tf_names.txt, cistromes_direct.h5ad, cistromes_extended.h5ad
    jobid: 6
    reason: Missing output files: cistromes_direct.h5ad, cistromes_extended.h5ad, tf_names.txt; Input files updated by another job: dem_results.hdf5, ctx_results.hdf5
    resources: tmpdir=/tmp


[Sun Mar 29 16:40:11 2026]
localrule get_search_space:
    input: ACC_GEX.fixed3.h5mu, genome_annotation.tsv, chromsizes.tsv
    output: search_space.tsv
    jobid: 11
    reason: Updated input files: ACC_GEX.fixed3.h5mu
    resources: tmpdir=/tmp



2026-03-29 16:40:14,880 SCENIC+      INFO     Reading multiome MuData.
2026-03-29 16:40:14,981 SCENIC+      INFO     Reading data
2026-03-29 16:40:33,191 SCENIC+      INFO     Getting cistromes.
2026-03-29 16:40:33,326 Get search space INFO     Extending search space to:
						150000 bp downstream of the end of the gene.
						150000 bp upstream of the start of the gene.
2026-03-29 16:40:33,406 Get search space INFO     Intersecting with regions.
2026-03-29 16:40:33,406 Get search space INFO     reg_df shape: (50187, 4), ext shape: (28547, 9)
2026-03-29 16:40:33,582 SCENIC+      INFO     Found 158 TFs.
2026-03-29 16:40:33,582 SCENIC+      INFO     Saving TF names to: tf_names.txt
2026-03-29 16:40:33,587 SCENIC+      INFO     Writing direct cistromes to: cistromes_direct.h5ad
2026-03-29 16:40:33,597 SCENIC+      INFO     Writing extended cistromes to: cistromes_extended.h5ad


... (25 outputs left out)

2026-03-29 17:32:57,732 SCENIC+      INFO     Reading data.
2026-03-29 17:33:05,609 SCENIC+      WARNING  Empty eRegulons file eRegulon_direct.tsv — writing empty AUCell output.


[Sun Mar 29 17:33:06 2026]
Finished job 3.
10 of 12 steps (83%) done
Select jobs to execute...
Execute 1 jobs...

[Sun Mar 29 17:33:06 2026]
localrule scplus_mudata:
    input: ACC_GEX.fixed3.h5mu, AUCell_direct.h5mu, AUCell_extended.h5mu, eRegulon_direct.tsv, eRegulons_extended.tsv
    output: scplusmdata.h5mu
    jobid: 1
    reason: Missing output files: scplusmdata.h5mu; Input files updated by another job: eRegulon_direct.tsv, AUCell_direct.h5mu, eRegulons_extended.tsv, AUCell_extended.h5mu
    resources: tmpdir=/tmp



2026-03-29 17:33:09,287 SCENIC+      INFO     Reading multiome MuData.
2026-03-29 17:33:17,009 SCENIC+      INFO     Reading AUC values.
2026-03-29 17:33:17,122 SCENIC+      INFO     Reading eRegulon metadata.
2026-03-29 17:33:17,142 SCENIC+      INFO     Generating MuData object.
2026-03-29 17:33:17,210 SCENIC+      INFO     Writing file to scplusmdata.h5mu


[Sun Mar 29 17:33:25 2026]
Finished job 1.
11 of 12 steps (92%) done
Select jobs to execute...
Execute 1 jobs...

[Sun Mar 29 17:33:25 2026]
localrule all:
    input: scplusmdata.h5mu
    jobid: 0
    reason: Input files updated by another job: scplusmdata.h5mu
    resources: tmpdir=/tmp

[Sun Mar 29 17:33:25 2026]
Finished job 0.
12 of 12 steps (100%) done
Complete log: .snakemake/log/2026-03-29T163917.751441.snakemake.log


In [7]:
# ================================================================
# CELL 18 — SCENIC+ DIAGNOSTICS + MG ANALYSIS
#
# Structure:
#   SECTION 1  — Pipeline file checks
#   SECTION 3  — cisTopic object
#   SECTION 4  — RNA AnnData + sample confounding check
#   SECTION 6  — DEM / cisTarget / Cistromes
#   SECTION 7  — tf_to_gene + region_to_gene adjacencies
#   SECTION 8  — eRegulon summary
#   SECTION 9  — Final scplus MuData
#   ── MG ANALYSIS ──────────────────────────────────────────────
#   PART A  — Activated vs Resting MG  (eRegulon AUC + RNA + ATAC)
#   PART B  — All MG vs Background
#   PART C  — Combined TF evidence matrix (both comparisons)
#   PART D  — Extended TF summary (no eRegulon)
#   PART E  — Top eRegulon TF regulation audit (data-driven)
#   PART F  — Top pathway genes (data-driven)
#   PART G  — TF→gene network + hub genes
#   PART H  — JASPAR motif enrichment summary (data-driven)
#   PART I  — Output file summary
# ================================================================
import pandas as pd
import numpy as np
import anndata
import mudata
import pickle
import h5py
import scipy.sparse as sp
import scanpy as sc
import networkx as nx
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────
BASE   = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/")
INDIR  = BASE / "scenicplus_input"
WD     = BASE / "scplus_pipeline/Snakemake"
OUTDIR = BASE / "scenic_outputs"
OUTDIR.mkdir(exist_ok=True)

# ── Config (edit here) ────────────────────────────────────────────
MIN_EREG_TARGETS   = 5    # eRegulons with fewer targets excluded from priority analysis
TOP_N_GENES        = 20   # genes shown in per-direction tables
IMP_PCT            = 95   # tf_to_gene importance percentile for network edges
N_EXTENDED_TFS     = 10   # top tf_to_gene TFs not in eRegulons to include
TOP_TF_AUDIT       = 3    # top eRegulon TFs to run the regulation audit (PART E)
TOP_PATHWAY_GENES  = 25   # top DEGs to annotate in pathway summary (PART F)
DEG_FC_THRESH      = 0.25 # |log2FC| threshold for TF→target overlap counts

# ── MG cell-state labels (match obs["celltype"] values exactly) ───
ACT_LABEL  = "activated MG"
REST_LABEL = "resting MG"
MG_LABELS  = [ACT_LABEL, REST_LABEL]

# ── hdf5plugin: import once, reuse for both DEM and CTX ───────────
try:
    import hdf5plugin
except ImportError:
    pass  # Optional compression plugin — gracefully absent is fine

# ── Helpers ───────────────────────────────────────────────────────
def section(title):
    print(f"\n{'─'*70}")
    print(f"  {title}")
    print(f"{'─'*70}")

def file_check(label, path):
    p      = Path(path)
    exists = p.exists()
    size   = p.stat().st_size / 1e6 if exists else 0
    print(f"  {'✓' if exists else '✗'} {label:<45} {size:>8.1f} MB")
    return exists

print("=" * 70)
print("SCENIC+ DIAGNOSTIC + MG ANALYSIS")
print("=" * 70)

# ════════════════════════════════════════════════════════════════════
# SECTION 1 — Pipeline file checks
# ════════════════════════════════════════════════════════════════════
section("1. PIPELINE FILES")
all_files = {
    "cisTopic pkl":                  INDIR / "atac_cistopic_selected_model_chrstr.pkl",
    "RNA h5ad":                      INDIR / "rna_scanpy_preprocessed.h5ad",
    "motifs.filtered.tbl":           INDIR / "motifs.filtered.tbl",
    "scores feather (DEM)":          INDIR / "gallus_consensus_chicken_expr.regions_vs_motifs.scores.feather",
    "rankings feather (CTX)":        INDIR / "gallus_consensus_chicken_expr.regions_vs_motifs.rankings.feather",
    "region_sets folder":            INDIR / "region_sets",
    "ACC_GEX metacell h5mu":         WD    / "ACC_GEX.fixed3.h5mu",
    "dem_results.hdf5":              WD    / "dem_results.hdf5",
    "ctx_results.hdf5":              WD    / "ctx_results.hdf5",
    "cistromes_extended.h5ad":       WD    / "cistromes_extended.h5ad",
    "cistromes_direct.h5ad":         WD    / "cistromes_direct.h5ad",
    "tf_to_gene_adj.tsv":            WD    / "tf_to_gene_adj.tsv",
    "region_to_gene_adj.tsv":        WD    / "region_to_gene_adj.tsv",
    "eRegulons_extended.tsv":        WD    / "eRegulons_extended.tsv",
    "scplusmdata.h5mu":              WD    / "scplusmdata.h5mu",
    "da_rna_full.csv":               BASE  / "da_rna_full.csv",
    "da_combined_full.csv":          BASE  / "da_combined_full.csv",
    "JASPAR reconciled enrichment":  BASE  / "enriched_motifs_scenic_reconciled.csv",
    "JASPAR human-only enrichment":  BASE  / "enriched_motifs_sorted.csv",
    "JASPAR comparison summary":     BASE  / "motif_comparison_summary.csv",
}
for label, path in all_files.items():
    file_check(label, path)

# ════════════════════════════════════════════════════════════════════
# SECTION 3 — cisTopic object
# ════════════════════════════════════════════════════════════════════
section("3. CISTOPIC OBJECT")
with open(INDIR / "atac_cistopic_selected_model_chrstr.pkl", "rb") as f:
    cistopic = pickle.load(f)
print(f"  Type: {type(cistopic).__name__}")
if hasattr(cistopic, "cell_data"):
    print(f"  Cells: {len(cistopic.cell_data)}")
    for col in ["sample", "celltype"]:
        if col in cistopic.cell_data.columns:
            print(f"  {col}: {cistopic.cell_data[col].value_counts().to_dict()}")
if hasattr(cistopic, "region_data"):
    print(f"  Regions: {len(cistopic.region_data)}")
    print(f"  Sample region names: {[str(x) for x in cistopic.region_data.index[:3].tolist()]}")
if hasattr(cistopic, "selected_model") and hasattr(cistopic.selected_model, "cell_topic"):
    shape = cistopic.selected_model.cell_topic.shape
    ct    = pd.DataFrame(cistopic.selected_model.cell_topic)
    print(f"  cell_topic shape: {shape}  →  Topics: {min(shape)}")
    print(f"  Topic prob: mean={ct.values.mean():.4f}  std={ct.values.std():.4f}  max={ct.values.max():.4f}")

# ════════════════════════════════════════════════════════════════════
# SECTION 4 — RNA AnnData + sample confounding
# ════════════════════════════════════════════════════════════════════
section("4. RNA ANNDATA + SAMPLE CONFOUNDING")
import warnings as _w
with _w.catch_warnings():
    _w.simplefilter("ignore")
    rna = sc.read_h5ad(INDIR / "rna_scanpy_preprocessed.h5ad")
print(f"  Shape: {rna.shape}  (cells × genes)")
X_s    = rna.X[:200, :200]
X_a    = X_s.toarray() if sp.issparse(X_s) else np.array(X_s)
is_int = np.allclose(X_a[X_a > 0], np.round(X_a[X_a > 0]))
print(f"  X dtype: {rna.X.dtype} | Integer counts: {is_int}")
if not is_int:
    print("  Note: non-integer — RNA includes imputed ATAC cells (Seurat TransferData).")
print(f"  obs columns: {rna.obs.columns.tolist()}")
for col in rna.obs.columns:
    vc = rna.obs[col].dropna().value_counts()
    if 0 < len(vc) <= 20:
        print(f"\n  {col}:")
        print(vc.to_string())
    elif len(vc) > 20:
        print(f"\n  {col}: {len(vc)} unique values (top 3: {vc.head(3).index.tolist()})")

# Filter to MG cells using config labels
mg_cells = rna.obs[rna.obs["celltype"].isin(MG_LABELS)].copy()
print(f"\n  MG cell counts:")
mg_vc = mg_cells["celltype"].value_counts()
print(mg_vc[mg_vc > 0].to_string())

if "sample" in rna.obs.columns:
    cross = mg_cells.groupby(["celltype", "sample"]).size().unstack(fill_value=0)
    print(f"\n  MG cells by sample (confounding check):")
    print(cross.to_string())

    confounding_flags = {}
    for ct in MG_LABELS:
        sub = mg_cells[mg_cells["celltype"] == ct]
        if not len(sub):
            continue
        dominant_frac = sub["sample"].value_counts().iloc[0] / len(sub)
        dominant_samp = sub["sample"].value_counts().index[0]
        confounding_flags[ct] = (dominant_frac, dominant_samp)

    max_conf = max(v[0] for v in confounding_flags.values()) if confounding_flags else 0
    if max_conf >= 1.0:
        print("\n" + "!" * 70)
        print("  !! COMPLETE SAMPLE CONFOUNDING — CRITICAL INTERPRETATION WARNING !!")
        print("!" * 70)
        print(f"  {ACT_LABEL} and {REST_LABEL} are 100% separated by sample (NM vs SS).")
        print("  ALL DEGs, DA peaks, and eRegulon AUC differences are statistically")
        print("  indistinguishable from sample/batch effects. The biology is expected")
        print("  (NMDA drives MG activation) but the comparison CANNOT distinguish")
        print("  cell-state from treatment-batch signals without orthogonal validation.")
        print("!" * 70)
    elif max_conf > 0.9:
        print("\n  ⚠ STRONG CONFOUNDING: MG states are nearly perfectly separated by sample.")

    for ct, (dominant_frac, dominant_samp) in confounding_flags.items():
        flag = ("COMPLETE" if dominant_frac >= 1.0 else
                "CAUTION"  if dominant_frac > 0.75 else "OK")
        print(f"  {flag}: {ct} — {dominant_frac:.1%} from sample '{dominant_samp}'")

if "prediction.score.max" in mg_cells.columns:
    scores = mg_cells["prediction.score.max"].dropna()
    if len(scores) > 0:
        print(f"\n  Label transfer confidence (prediction.score.max, n={len(scores)}):")
        print(mg_cells[mg_cells["prediction.score.max"].notna()]
              .groupby("celltype")["prediction.score.max"].describe().round(3).to_string())
        print(f"  Cells with score < 0.5: {(scores < 0.5).sum()}")
    else:
        print("\n  Label transfer confidence: scores not present in h5ad (lost during export).")
        print("  Check prediction.score.max in the original rna_prePlot.rds object.")

# ════════════════════════════════════════════════════════════════════
# SECTION 6 — DEM / cisTarget / Cistromes
#
# SCENIC+ writes motif enrichment results in pytables format:
#   grp / motif_enrichment / table /
#       field_N   — blosc-compressed column data (one dataset per column)
#       metadata  — HDF5 attributes include FIELD_N_NAME → column name
#       _i_table  — index (ignored)
# We reconstruct the DataFrame by reading FIELD_N_NAME from metadata
# attributes and pairing each with its field_N dataset.
# ════════════════════════════════════════════════════════════════════
import warnings as _warnings

def _decode_col(series):
    """Cast a series to Python-native types safe for to_string()."""
    if series.dtype.kind == "S":
        return series.str.decode("utf-8", errors="replace")
    if series.dtype == object:
        def _v(v):
            if isinstance(v, (bytes, np.bytes_)):
                return v.decode("utf-8", errors="replace")
            if isinstance(v, np.ndarray):
                return str(v.tolist())
            return v
        return series.map(_v)
    return series

def _read_pytables_group(tbl_grp):
    """
    Read one pytables 'table' group into a DataFrame.
    Column names come from FIELD_N_NAME attributes on the metadata dataset.
    Data comes from field_N datasets (blosc-compressed).
    Falls back to reading any 1-D array-shaped dataset if metadata absent.
    """
    # --- Strategy 1: metadata attributes carry FIELD_N_NAME ---
    col_names = {}
    if "metadata" in tbl_grp:
        meta = tbl_grp["metadata"]
        for attr_name, attr_val in meta.attrs.items():
            if attr_name.startswith("FIELD_") and attr_name.endswith("_NAME"):
                idx = int(attr_name.split("_")[1])
                if isinstance(attr_val, bytes):
                    attr_val = attr_val.decode("utf-8")
                col_names[idx] = attr_val

    data = {}
    if col_names:
        for idx, col in col_names.items():
            field_key = f"field_{idx}"
            if field_key in tbl_grp:
                try:
                    arr = tbl_grp[field_key][()]
                    if arr.ndim > 1:
                        arr = arr[:, 0] if arr.shape[1] == 1 else np.array([str(r.tolist()) for r in arr])
                    if arr.dtype.kind == "S":
                        arr = np.char.decode(arr, "utf-8")
                    data[col] = arr
                except Exception:
                    pass

    # --- Strategy 2: fallback — read all 1-D datasets, use dataset name ---
    if not data:
        for k in tbl_grp.keys():
            if k in ("metadata", "_i_table"):
                continue
            try:
                arr = tbl_grp[k][()]
                if not isinstance(arr, np.ndarray) or arr.ndim == 0:
                    continue
                if arr.ndim > 1:
                    arr = arr[:, 0] if arr.shape[1] == 1 else None
                if arr is None:
                    continue
                if arr.dtype.kind == "S":
                    arr = np.char.decode(arr, "utf-8")
                data[k] = arr
            except Exception:
                pass

    if not data:
        return None
    min_len = min(len(v) for v in data.values())
    df = pd.DataFrame({k: v[:min_len] for k, v in data.items()})
    for col in df.columns:
        df[col] = _decode_col(df[col])
    for col in df.select_dtypes(include="object").columns:
        df[col] = pd.to_numeric(df[col], errors="ignore")
    return df

def _read_hdf5_motif_group(h5_path, max_topics=None):
    """Read SCENIC+ HDF5 motif enrichment results. Returns list of DataFrames."""
    dfs = []
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())
        for key in (keys[:max_topics] if max_topics else keys):
            grp      = f[str(key)]
            grp_keys = list(grp.keys())
            me_key   = next((k for k in grp_keys
                             if "motif" in k.lower() or "enrich" in k.lower()), None)
            if me_key is None:
                continue
            me = grp[me_key]
            df = None

            if "table" in me.keys():
                tbl = me["table"]
                if hasattr(tbl, "keys"):
                    # pytables group layout (most common in SCENIC+ ≥1.0)
                    df = _read_pytables_group(tbl)
                else:
                    # flat structured numpy array (older pytables)
                    try:
                        raw = tbl[()]
                        if hasattr(raw.dtype, "names") and raw.dtype.names:
                            df = pd.DataFrame(raw)
                            for col in df.columns:
                                df[col] = _decode_col(df[col])
                    except Exception:
                        pass

            # fallback: fields are direct children of motif_enrichment group
            if df is None:
                data = {}
                for k in me.keys():
                    if k == "table":
                        continue
                    try:
                        item = me[k]
                        if not hasattr(item, "shape"):
                            continue
                        arr = item[()]
                        if arr.ndim > 1:
                            arr = arr[:, 0] if arr.shape[1] == 1 else np.array([str(r.tolist()) for r in arr])
                        if arr.dtype.kind == "S":
                            arr = np.char.decode(arr, "utf-8")
                        data[k] = arr
                    except Exception:
                        pass
                if data:
                    min_len = min(len(v) for v in data.values())
                    df = pd.DataFrame({k: v[:min_len] for k, v in data.items()})

            if df is not None and len(df):
                df["topic"] = key
                dfs.append(df)
    return dfs

def _print_hdf5_top(dfs, label):
    """Concat parsed HDF5 motif DataFrames and display top results."""
    dfs = [d for d in dfs if len(d.columns) > 1]
    if not dfs:
        print(f"  {label}: no enrichment data found")
        return None
    df = pd.concat(dfs, ignore_index=True)
    print(f"  {label}: {len(df):,} entries  |  columns: {df.columns.tolist()}")
    sort_candidates = ["Log2FC","log2FC","NES","nes","Adjusted_pval","pval","adj_pval",
                       "adjusted_pvalue","AUC","auc"]
    num_cols = [c for c in sort_candidates
                if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if num_cols:
        sort_col   = num_cols[0]
        ascending  = "pval" in sort_col.lower() or "adj" in sort_col.lower()
        display_df = df.sort_values(sort_col, ascending=ascending).head(15).copy()
        for col in display_df.select_dtypes(include="object").columns:
            display_df[col] = display_df[col].astype(str)
        print(f"\n  Top 15 by {sort_col}:")
        print(display_df.to_string(index=False))
    else:
        display_df = df.head(10).copy()
        for col in display_df.select_dtypes(include="object").columns:
            display_df[col] = display_df[col].astype(str)
        print(f"  (no recognised numeric sort column — first 10 rows):")
        print(display_df.to_string(index=False))
    return df

section("6. DEM RESULTS")
dem_path = WD / "dem_results.hdf5"
all_dem  = None
if dem_path.exists():
    print(f"  File size: {dem_path.stat().st_size/1e6:.1f} MB")
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        all_dem = _print_hdf5_top(_read_hdf5_motif_group(dem_path), "DEM")
else:
    print("  Not yet generated")

section("6b. CISTARGET RESULTS")
ctx_path = WD / "ctx_results.hdf5"
all_ctx  = None
if ctx_path.exists():
    print(f"  File size: {ctx_path.stat().st_size/1e6:.1f} MB")
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        all_ctx = _print_hdf5_top(_read_hdf5_motif_group(ctx_path, max_topics=3),
                                   "CTX (first 3 topics)")
else:
    print("  Not yet generated")

section("6c. CISTROMES")
for fname, label in [("cistromes_extended.h5ad", "Extended (Orthology)"),
                      ("cistromes_direct.h5ad",   "Direct")]:
    cpath = WD / fname
    if cpath.exists():
        import warnings as _w
        with _w.catch_warnings():
            _w.simplefilter("ignore")
            cist = anndata.read_h5ad(str(cpath))
        print(f"\n  {label}: {cist.n_vars} TFs × {cist.n_obs} regions")
        if cist.n_vars > 0:
            X_c      = cist.X.toarray() if sp.issparse(cist.X) else np.array(cist.X)
            tf_stats = (pd.DataFrame({"TF": cist.var_names,
                                      "n_regions": (X_c > 0).sum(axis=0)})
                        .sort_values("n_regions", ascending=False))
            print(f"  All TFs: {sorted(cist.var_names.tolist())}")
            print(f"\n  Top 15 TFs by enriched region count:")
            print(tf_stats.head(15).to_string(index=False))
        else:
            print("  Empty — no TFs passed motif enrichment thresholds")
            print("  (Direct empty = expected for non-model organism)")
    else:
        print(f"\n  {label}: not yet generated")

# ════════════════════════════════════════════════════════════════════
# SECTION 7 — tf_to_gene + region_to_gene
# ════════════════════════════════════════════════════════════════════
section("7. TF_TO_GENE ADJACENCIES")
t2g_path = WD / "tf_to_gene_adj.tsv"
tf2g     = None
if t2g_path.exists():
    tf2g = pd.read_csv(t2g_path, sep="\t")
    print(f"  Shape: {tf2g.shape}  |  TFs: {tf2g['TF'].nunique()}  |  Targets: {tf2g['target'].nunique()}")
    print(f"  Importance: mean={tf2g['importance'].mean():.4f}  max={tf2g['importance'].max():.4f}")
    print(f"  Rho: mean={tf2g['rho'].mean():.4f}  pos={(tf2g['rho'] > 0).mean()*100:.1f}%")
    tf_imp = (tf2g.groupby("TF")["importance"]
              .agg(["mean","max","count"])
              .sort_values("mean", ascending=False))
    tf_imp.columns = ["mean_imp","max_imp","n_targets"]
    print(f"\n  All TFs by mean importance:")
    print(tf_imp.to_string())
    print(f"\n  Top 10 TF→gene links by importance:")
    print(tf2g.nlargest(10, "importance").to_string(index=False))
else:
    print("  Not yet generated")

section("7b. REGION_TO_GENE ADJACENCIES")
r2g_path = WD / "region_to_gene_adj.tsv"
if r2g_path.exists():
    r2g = pd.read_csv(r2g_path, sep="\t")
    print(f"  Shape: {r2g.shape}  |  Genes: {r2g['target'].nunique()}  |  Regions: {r2g['region'].nunique()}")

    placed_mask = ~r2g["region"].str.contains("JAENSK|scaffold|Un_", na=False, case=False)
    n_unplaced  = (~placed_mask).sum()
    if n_unplaced > 0:
        print(f"  WARNING: {n_unplaced:,} links ({n_unplaced/len(r2g):.1%}) involve unplaced "
              f"scaffold regions — inflated importance=1.0, near-zero rho. Filtered below.")
    r2g_placed = r2g[placed_mask]
    print(f"  Placed-contig links: {len(r2g_placed):,}  |  "
          f"Genes: {r2g_placed['target'].nunique()}  |  Regions: {r2g_placed['region'].nunique()}")
    print(f"  Importance (placed): mean={r2g_placed['importance'].mean():.4f}  "
          f"max={r2g_placed['importance'].max():.4f}")
    print(f"  Rho (placed): mean={r2g_placed['rho'].mean():.4f}  "
          f"pos={(r2g_placed['rho'] > 0).mean()*100:.1f}%")
    if "Distance" in r2g_placed.columns:
        print(f"  Distance (placed): mean={r2g_placed['Distance'].mean():.0f}bp  "
              f"median={r2g_placed['Distance'].median():.0f}bp")
    disp = [c for c in ["target","region","importance","rho","Distance"] if c in r2g.columns]
    print(f"\n  Top 10 region→gene links (placed contigs only):")
    print(r2g_placed.nlargest(10, "importance")[disp].to_string(index=False))
    print(f"\n  Top 15 genes by mean region importance (placed contigs only):")
    print(r2g_placed.groupby("target")["importance"].mean()
          .sort_values(ascending=False).head(15).to_string())
else:
    print("  Not yet generated")

# ════════════════════════════════════════════════════════════════════
# SECTION 8 — eRegulon summary
# ════════════════════════════════════════════════════════════════════
section("8. EREGULONS (extended)")
ereg_path = WD / "eRegulons_extended.tsv"
eregs     = None
if ereg_path.exists():
    eregs = pd.read_csv(ereg_path, sep="\t")
    print(f"  Shape: {eregs.shape}  |  TFs: {eregs['TF'].nunique()}  "
          f"|  Target genes: {eregs['Gene'].nunique()}  |  Regions: {eregs['Region'].nunique()}")
    tgt_counts = eregs.groupby("TF")["Gene"].nunique().sort_values(ascending=False)
    print(f"\n  TF target gene counts:")
    print(tgt_counts.to_string())
    small = tgt_counts[tgt_counts < MIN_EREG_TARGETS]
    if len(small):
        print(f"\n  Small eRegulons (<{MIN_EREG_TARGETS} targets — excluded from priority analysis):")
        print(f"    {small.index.tolist()}")
    if "regulation" in eregs.columns:
        print(f"\n  Regulation modes (all links):")
        reg_map = {1: "activating", -1: "repressing", 0: "unknown"}
        reg_cts = eregs["regulation"].value_counts().to_dict()
        for k, n in sorted(reg_cts.items()):
            print(f"    {reg_map.get(k,k)} ({k}): {n:,}  ({n/len(eregs):.1%})")
    if "triplet_rank" in eregs.columns:
        print(f"\n  Triplet rank: mean={eregs['triplet_rank'].mean():.2f}  "
              f"min={eregs['triplet_rank'].min():.2f}")
    print(f"\n  First 5 rows:")
    print(eregs.head(5).to_string(index=False))
else:
    print("  Not yet generated")

# ════════════════════════════════════════════════════════════════════
# SECTION 9 — Final scplus MuData  (loaded once, reused in MG analysis)
# ════════════════════════════════════════════════════════════════════
section("9. FINAL SCPLUS MUDATA")
scp_path     = WD / "scplusmdata.h5mu"
scplus_mdata = None
if scp_path.exists():
    import warnings as _w
    with _w.catch_warnings():
        _w.simplefilter("ignore")
        scplus_mdata = mudata.read_h5mu(str(scp_path))   # single load — reused below
    print(f"  Total metacells: {scplus_mdata.n_obs}  |  "
          f"obs columns: {scplus_mdata.obs.columns.tolist()}")
    print(f"\n  Modalities:")
    for k, v in scplus_mdata.mod.items():
        print(f"    {k}: {v.shape}")
    print(f"\n  uns keys: {list(scplus_mdata.uns.keys())}")
    for uns_key in ["extended_e_regulon_metadata", "direct_e_regulon_metadata"]:
        if uns_key in scplus_mdata.uns:
            em = scplus_mdata.uns[uns_key]
            print(f"\n  {uns_key}: {em.shape}")
            print(em.head(3).to_string(index=False))
else:
    print("  Not yet generated")

# ════════════════════════════════════════════════════════════════════
# Load MG analysis inputs
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("MG ANALYSIS")
print("=" * 70)

da_rna_path  = BASE / "da_rna_full.csv"
da_comb_path = BASE / "da_combined_full.csv"
if not da_rna_path.exists():
    print("da_rna_full.csv not found — run R analysis first. Stopping.")
    raise SystemExit

da_rna      = pd.read_csv(da_rna_path).set_index("gene")
da_combined = (pd.read_csv(da_comb_path).set_index("gene")
               if da_comb_path.exists() else pd.DataFrame())

jaspar_rec  = (pd.read_csv(BASE / "enriched_motifs_scenic_reconciled.csv")
               if (BASE / "enriched_motifs_scenic_reconciled.csv").exists() else None)
jaspar_old  = (pd.read_csv(BASE / "enriched_motifs_sorted.csv")
               if (BASE / "enriched_motifs_sorted.csv").exists() else None)
jaspar_comp = (pd.read_csv(BASE / "motif_comparison_summary.csv")
               if (BASE / "motif_comparison_summary.csv").exists() else None)

# eRegulon filters
ereg_sizes      = (eregs.groupby("TF")["Gene"].nunique()
                   if eregs is not None else pd.Series(dtype=int))
reliable_tfs    = ereg_sizes[ereg_sizes >= MIN_EREG_TARGETS].index.tolist()
eregs_rel       = (eregs[eregs["TF"].isin(reliable_tfs)].copy()
                   if eregs is not None else pd.DataFrame())
small_eregs     = ereg_sizes[ereg_sizes < MIN_EREG_TARGETS].index.tolist()
ereg_target_set = set(eregs["Gene"].str.upper()) if eregs is not None else set()
ereg_target_map = (eregs.groupby("Gene")["TF"]
                   .apply(lambda x: "|".join(x.unique()))
                   .to_dict()) if eregs is not None else {}

print(f"\neRegulon TFs total: {ereg_sizes.shape[0]}  "
      f"reliable (≥{MIN_EREG_TARGETS} targets): {len(reliable_tfs)}  "
      f"small (<{MIN_EREG_TARGETS}): {len(small_eregs)}")
if small_eregs:
    print(f"  Small eRegulons: {small_eregs}")

# Extended TFs: top tf_to_gene TFs not in any eRegulon
if tf2g is not None:
    tf2g_mean    = tf2g.groupby("TF")["importance"].mean().sort_values(ascending=False)
    extended_tfs = [t for t in tf2g_mean.index
                    if t not in ereg_sizes.index][:N_EXTENDED_TFS]
else:
    tf2g_mean    = pd.Series(dtype=float)
    extended_tfs = []

# AUC matrix — already loaded as scplus_mdata above (no second read)
auc_m = act_auc = rest_auc = mg_auc = bg_auc = None
if scplus_mdata is not None:
    raw_obs = scplus_mdata.obs.copy()
    new_obs = pd.DataFrame(index=raw_obs.index)
    added   = set()
    for col in raw_obs.columns:
        clean = col
        for key in ["celltype", "cell_type"]:
            if col.endswith(":" + key) or col == key:
                clean = key; break
        if clean not in added:
            new_obs[clean] = raw_obs[col].values; added.add(clean)

    auc_m  = scplus_mdata["extended_gene_based_AUC"]
    X_auc  = auc_m.X.toarray() if sp.issparse(auc_m.X) else np.array(auc_m.X)
    auc_df = (pd.DataFrame(X_auc, index=auc_m.obs_names, columns=auc_m.var_names)
              .join(new_obs[["celltype"]], how="left"))
    act_auc  = auc_df[auc_df["celltype"] == ACT_LABEL].drop(columns="celltype")
    rest_auc = auc_df[auc_df["celltype"] == REST_LABEL].drop(columns="celltype")
    mg_auc   = auc_df[auc_df["celltype"].isin(MG_LABELS)].drop(columns="celltype")
    bg_auc   = auc_df[~auc_df["celltype"].isin(MG_LABELS)].drop(columns="celltype")
    print(f"\nAUC metacells — {ACT_LABEL}: {len(act_auc)}  "
          f"{REST_LABEL}: {len(rest_auc)}  background: {len(bg_auc)}")

# ════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ════════════════════════════════════════════════════════════════════

def compute_tf_stats(g1_auc, g2_auc, label1, label2):
    """
    Mean AUC across all eRegulon columns per TF → Mann-Whitney U.
    Avoids best-column selection bias (anti-conservative p-values).
    Extended TFs get RNA/ATAC stats only — no AUC column exists for them.
    """
    rows    = []
    all_tfs = reliable_tfs + [t for t in extended_tfs if t not in reliable_tfs]
    for tf in all_tfs:
        in_ereg             = tf in reliable_tfs
        delta, pval, ecols  = np.nan, np.nan, ""
        if in_ereg and auc_m is not None:
            matches = [c for c in auc_m.var_names if c.startswith(tf + "_")]
            if matches:
                g1_mean = g1_auc[matches].mean(axis=1).values
                g2_mean = g2_auc[matches].mean(axis=1).values
                delta   = g1_mean.mean() - g2_mean.mean()
                _, pval = stats.mannwhitneyu(g1_mean, g2_mean, alternative="two-sided")
                ecols   = "|".join(matches)
        rna_r  = da_rna.loc[tf]      if tf in da_rna.index else None
        comb_r = (da_combined.loc[tf]
                  if len(da_combined) and tf in da_combined.index else None)
        t2g_s  = tf2g[tf2g["TF"] == tf] if tf2g is not None else pd.DataFrame()
        rows.append(dict(
            TF=tf, in_eRegulon=in_ereg, eRegulon_cols=ecols,
            delta_AUC=round(delta, 4) if not np.isnan(delta) else np.nan,
            p_val=round(pval, 6)      if not np.isnan(pval)  else np.nan,
            direction=(label1 if (not np.isnan(delta) and delta > 0) else
                       label2 if (not np.isnan(delta) and delta <= 0) else
                       (label1 if (rna_r is not None and rna_r["avg_log2FC"] > 0)
                        else label2)),
            TF_RNA_log2FC=(round(float(rna_r["avg_log2FC"]), 3)
                           if rna_r is not None else np.nan),
            TF_RNA_padj=(round(float(rna_r["p_val_adj"]), 6)
                         if rna_r is not None else np.nan),
            TF_RNA_sig=(bool(rna_r["p_val_adj"] < 0.05) if rna_r is not None else False),
            TF_ATAC_log2FC=(round(float(comb_r["avg_log2FC.activity"]), 3)
                            if comb_r is not None else np.nan),
            TF_ATAC_sig=(bool(comb_r["p_val_adj.activity"] < 0.05)
                         if comb_r is not None else False),
            n_ereg_targets=int(ereg_sizes.get(tf, 0)),
            t2g_n=len(t2g_s),
            t2g_mean_imp=(round(t2g_s["importance"].mean(), 3)
                          if len(t2g_s) else np.nan),
        ))
    df        = pd.DataFrame(rows)
    ereg_mask = df["in_eRegulon"] & df["p_val"].notna()
    if ereg_mask.any():
        _, q, _, _ = multipletests(df.loc[ereg_mask, "p_val"], method="fdr_bh")
        df.loc[ereg_mask, "q_val"] = q.round(6)
    if "q_val" not in df.columns:
        df["q_val"] = np.nan
    df["AUC_sig"]  = df["q_val"] < 0.05
    df["n_layers"] = (df["AUC_sig"].astype(int) +
                      df["TF_RNA_sig"].astype(int) +
                      df["TF_ATAC_sig"].astype(int))
    ereg_df = df[df["in_eRegulon"]].sort_values("delta_AUC", ascending=False)
    ext_df  = df[~df["in_eRegulon"]].sort_values("TF_RNA_log2FC", ascending=False,
                                                   na_position="last")
    return pd.concat([ereg_df, ext_df], ignore_index=True)


def build_gene_table():
    gt = da_rna[["avg_log2FC","p_val_adj"]].rename(
        columns={"avg_log2FC":"RNA_log2FC","p_val_adj":"RNA_padj"})
    if len(da_combined):
        at = da_combined[["avg_log2FC.activity","p_val_adj.activity"]].rename(
            columns={"avg_log2FC.activity":"ATAC_log2FC","p_val_adj.activity":"ATAC_padj"})
        gt = gt.join(at, how="outer")
    else:
        gt["ATAC_log2FC"] = np.nan
        gt["ATAC_padj"]   = np.nan
    gt["RNA_sig"]        = gt["RNA_padj"]  < 0.05
    gt["ATAC_sig"]       = gt["ATAC_padj"] < 0.05
    gt["concordant"]     = np.sign(gt["RNA_log2FC"]) == np.sign(gt["ATAC_log2FC"])
    gt["n_sig"]          = gt["RNA_sig"].astype(int) + gt["ATAC_sig"].astype(int)
    gt["in_eReg_target"] = gt.index.str.upper().isin(ereg_target_set)
    gt["eReg_TFs"]       = gt.index.map(
        lambda g: ereg_target_map.get(g, ereg_target_map.get(g.upper(), "")))
    gt["is_eReg_TF"]     = (gt.index.isin(eregs["TF"].unique())
                             if eregs is not None else False)
    gt["is_reliable_TF"] = gt.index.isin(reliable_tfs)
    return gt


def print_tf_table(df, title):
    print(f"\n{title}")
    print("─" * 110)
    print(f"  {'TF':<11} {'eREG':>5} {'Dir':>6} {'dAUC':>9} {'q_AUC':>7} "
          f"{'RNA_FC':>8} {'RNA_p':>9} {'ATAC_FC':>9} "
          f"{'n_tgt':>6} {'t2g_n':>7} {'t2g_imp':>8} {'Lyrs':>5}")
    print("  " + "─" * 106)
    for _, r in df.iterrows():
        ereg_s = "✓" if r["in_eRegulon"] else "·"
        dauc_s = f"{r['delta_AUC']:+.4f}" if not np.isnan(r["delta_AUC"]) else "   -    "
        qval_s = (f"{r['q_val']:.4f}"
                  if not np.isnan(r.get("q_val", np.nan)) else "  -    ")
        rfc_s  = (f"{r['TF_RNA_log2FC']:+.3f}{'†' if r['TF_RNA_sig'] else ' '}"
                  if not np.isnan(r["TF_RNA_log2FC"]) else "   -   ")
        rp_s   = (f"{r['TF_RNA_padj']:.2e}"
                  if not np.isnan(r["TF_RNA_padj"]) else "  -      ")
        afc_s  = (f"{r['TF_ATAC_log2FC']:+.3f}{'†' if r['TF_ATAC_sig'] else ' '}"
                  if not np.isnan(r["TF_ATAC_log2FC"]) else "   -    ")
        nt_s   = str(int(r["n_ereg_targets"])) if r["n_ereg_targets"] > 0 else "-"
        ti_s   = (f"{r['t2g_mean_imp']:.3f}"
                  if not np.isnan(r["t2g_mean_imp"]) else "  -   ")
        print(f"  {r['TF']:<11} {ereg_s:>5} {r['direction']:>6} "
              f"{dauc_s:>9} {qval_s:>7} {rfc_s:>8} {rp_s:>9} "
              f"{afc_s:>9} {nt_s:>6} {int(r['t2g_n']):>7} {ti_s:>8} "
              f"{int(r['n_layers']):>5}")
    print(f"\n  ✓=eRegulon  ·=extended (tf_to_gene only)  †=sig(padj<0.05)  "
          f"dAUC=mean ΔAUC across all eReg cols  q_AUC=BH-corrected  "
          f"n_tgt=eRegulon targets  Lyrs=n sig layers")


def print_significance_summary(tf_df, label1, label2, label):
    print(f"\n  SIGNIFICANCE SUMMARY — {label}")
    print(f"  {'─'*65}")
    sig1 = tf_df[(tf_df["direction"] == label1) & tf_df["AUC_sig"] & tf_df["in_eRegulon"]]
    sig2 = tf_df[(tf_df["direction"] == label2) & tf_df["AUC_sig"] & tf_df["in_eRegulon"]]
    ml1  = tf_df[(tf_df["direction"] == label1) & (tf_df["n_layers"] >= 2)]
    ml2  = tf_df[(tf_df["direction"] == label2) & (tf_df["n_layers"] >= 2)]
    print(f"  eRegulon TFs sig in {label1} (q<0.05): {sig1['TF'].tolist()}")
    print(f"  eRegulon TFs sig in {label2} (q<0.05): {sig2['TF'].tolist()}")
    ml1_ereg = ml1[ml1["in_eRegulon"]];  ml1_ext = ml1[~ml1["in_eRegulon"]]
    ml2_ereg = ml2[ml2["in_eRegulon"]];  ml2_ext = ml2[~ml2["in_eRegulon"]]
    print(f"  Multi-layer {label1} eRegulon (≥2 layers): {ml1_ereg['TF'].tolist()}")
    if len(ml1_ext):
        print(f"  Multi-layer {label1} extended (RNA+ATAC only): {ml1_ext['TF'].tolist()}")
    print(f"  Multi-layer {label2} eRegulon (≥2 layers): {ml2_ereg['TF'].tolist()}")
    if len(ml2_ext):
        print(f"  Multi-layer {label2} extended (RNA+ATAC only): {ml2_ext['TF'].tolist()}")
    ext = tf_df[~tf_df["in_eRegulon"] & tf_df["TF_RNA_sig"]]
    if len(ext):
        ext1 = ext[ext["TF_RNA_log2FC"] > 0]
        ext2 = ext[ext["TF_RNA_log2FC"] < 0]
        if len(ext1):
            print(f"\n  Extended TFs with sig RNA in {label1}: "
                  f"{ext1.sort_values('TF_RNA_log2FC', ascending=False)['TF'].tolist()}")
        if len(ext2):
            print(f"  Extended TFs with sig RNA in {label2}: "
                  f"{ext2.sort_values('TF_RNA_log2FC')['TF'].tolist()}")
        print("  (Note: extended TFs ranked by whole-dataset tf_to_gene importance; "
              "RNA_FC is MG-specific)")
    return sig1, sig2


def print_gene_table(gt, up_label, dn_label):
    for lbl, mask, asc in [(up_label, gt["RNA_log2FC"] > 0, False),
                            (dn_label, gt["RNA_log2FC"] < 0, True)]:
        top = (gt[mask & gt["RNA_sig"]]
               .sort_values(["n_sig","RNA_log2FC"], ascending=[False, asc])
               .head(TOP_N_GENES))
        if not len(top):
            continue
        print(f"\n  {lbl} — top {TOP_N_GENES}")
        print(f"  {'Gene':<22} {'RNA_FC':>7} {'RNA_p':>9} "
              f"{'ATAC_FC':>8} {'Conc':>5} {'eReg':>5}  eRegulon_TFs")
        print("  " + "─" * 75)
        for gene, r in top.iterrows():
            atfc   = (f"{r['ATAC_log2FC']:+.2f}"
                      if not np.isnan(r.get("ATAC_log2FC", np.nan)) else "  -")
            tfmark = ("◆" if r["is_reliable_TF"] else
                      "◇" if r["is_eReg_TF"] else " ")
            print(f"  {tfmark}{gene:<21} {r['RNA_log2FC']:>+7.2f} "
                  f"{r['RNA_padj']:>9.2e} {atfc:>8} "
                  f"{'✓' if r.get('concordant') else '·':>5} "
                  f"{'✓' if r['in_eReg_target'] else '·':>5}  "
                  f"{r['eReg_TFs'][:30]}")
    print(f"\n  ◆=reliable eRegulon TF  ◇=small eRegulon TF  "
          f"Conc=RNA+ATAC concordant  eReg=eRegulon target")


def _top_t2g(gene, n=3):
    if tf2g is None:
        return ""
    hits = tf2g[tf2g["target"].str.upper() == gene.upper()]
    return "|".join(hits.nlargest(n, "importance")["TF"].tolist()) if len(hits) else ""


def _jasp_p(gene):
    if jaspar_rec is None:
        return ""
    row = jaspar_rec[jaspar_rec["motif.name"].str.upper() == gene.upper()]
    return f"  JASPAR_p={row.iloc[0]['pvalue']:.2e}" if len(row) else ""


def print_gene_stats(gene, gt, label=""):
    """Print a single gene's RNA/ATAC/eRegulon stats."""
    if gene not in gt.index:
        print(f"    {gene:<8}  not in DEG table")
        return
    r       = gt.loc[gene]
    sig     = "★" if not np.isnan(r.get("RNA_padj", np.nan)) and r["RNA_padj"] < 0.05 else " "
    in_ereg = "eReg" if gene in ereg_target_map else "    "
    print(f"    {gene:<8}{sig} FC={r['RNA_log2FC']:+.3f}  "
          f"padj={r['RNA_padj']:.2e}  {in_ereg}  "
          f"top_tf2g: {_top_t2g(gene)}{_jasp_p(gene)}")


# ════════════════════════════════════════════════════════════════════
# PART A — Activated vs Resting MG
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART A — ACTIVATED vs RESTING MG")
print("=" * 70)
print(f"  ⚠ CONFOUNDING NOTE: {ACT_LABEL} = NM sample; {REST_LABEL} = SS sample.")
print("  Results reflect NM vs SS comparison. Cell-state and treatment cannot")
print("  be separated without orthogonal validation.")

gt_ar = build_gene_table()

if act_auc is not None:
    tf_ar = compute_tf_stats(act_auc, rest_auc, "ACT", "REST")
    print_tf_table(tf_ar, "TFs — eRegulon TFs by ΔAUC | extended TFs by RNA FC")
    act_sig_ar, rest_sig_ar = print_significance_summary(tf_ar, "ACT", "REST",
                                                          "Activated vs Resting MG")
    tf_ar.to_csv(OUTDIR / "TF_act_vs_rest.csv", index=False)

    rna_act_genes  = set(gt_ar[gt_ar["RNA_sig"] & (gt_ar["RNA_log2FC"] >  DEG_FC_THRESH)]
                         .index.str.upper())
    rna_rest_genes = set(gt_ar[gt_ar["RNA_sig"] & (gt_ar["RNA_log2FC"] < -DEG_FC_THRESH)]
                         .index.str.upper())
    print(f"\n  TF→target DEG overlap (eRegulon TFs):")
    for _, r in tf_ar[tf_ar["in_eRegulon"]].iterrows():
        tgts   = set(eregs_rel[eregs_rel["TF"] == r["TF"]]["Gene"].str.upper())
        n_act  = len(tgts & rna_act_genes)
        n_rest = len(tgts & rna_rest_genes)
        if n_act > 0 or n_rest > 0:
            print(f"    {r['TF']:<12} act_targets={n_act:>4}  "
                  f"rest_targets={n_rest:>4}  (of {len(tgts)} total)")
else:
    tf_ar = pd.DataFrame()
    act_sig_ar, rest_sig_ar = pd.DataFrame(), pd.DataFrame()
    print("  scplus MuData not available — AUC analysis skipped")

print_gene_table(gt_ar, "Activated MG", "Resting MG")
gt_ar.to_csv(OUTDIR / "genes_act_vs_rest.csv")

# ════════════════════════════════════════════════════════════════════
# PART B — All MG vs Background
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART B — ALL MG vs BACKGROUND")
print("=" * 70)

if mg_auc is not None:
    tf_mb = compute_tf_stats(mg_auc, bg_auc, "MG", "BG")
    print_tf_table(tf_mb, "TFs — eRegulon TFs by ΔAUC | extended TFs by RNA FC")
    mg_sig, bg_sig = print_significance_summary(tf_mb, "MG", "BG", "MG vs Background")
    tf_mb.to_csv(OUTDIR / "TF_MG_vs_background.csv", index=False)

    if len(mg_sig):
        mg_tgt = (eregs_rel[eregs_rel["TF"].isin(mg_sig["TF"])]
                  [["TF","Gene","regulation"]].drop_duplicates())
        mg_tgt["RNA_log2FC"] = mg_tgt["Gene"].map(da_rna["avg_log2FC"])
        mg_tgt["RNA_padj"]   = mg_tgt["Gene"].map(da_rna["p_val_adj"])
        sig_tgt = mg_tgt[mg_tgt["RNA_padj"] < 0.05].sort_values("RNA_log2FC", ascending=False)
        if len(sig_tgt):
            print(f"\n  eRegulon targets of MG-enriched TFs (RNA sig, top 20):")
            print(sig_tgt[["TF","Gene","regulation","RNA_log2FC","RNA_padj"]]
                  .head(20).to_string(index=False))
        mg_tgt.to_csv(OUTDIR / "MG_vs_bg_TF_targets.csv", index=False)
else:
    tf_mb = pd.DataFrame()
    mg_sig, bg_sig = pd.DataFrame(), pd.DataFrame()
    print("  scplus MuData not available — AUC analysis skipped")

# ════════════════════════════════════════════════════════════════════
# PART C — Combined TF evidence matrix
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART C — COMBINED TF EVIDENCE MATRIX")
print("=" * 70)

comb = pd.DataFrame()
if len(tf_ar) and len(tf_mb):
    ar_idx = tf_ar.set_index("TF")[
        ["delta_AUC","q_val","direction","n_layers",
         "TF_RNA_log2FC","TF_RNA_padj","TF_RNA_sig",
         "in_eRegulon","n_ereg_targets","t2g_mean_imp"]]\
        .rename(columns={"delta_AUC":"dAUC_AR","q_val":"q_AR",
                         "direction":"dir_AR","n_layers":"lyrs_AR"})
    mb_idx = tf_mb.set_index("TF")[["delta_AUC","q_val","direction"]]\
        .rename(columns={"delta_AUC":"dAUC_MB","q_val":"q_MB","direction":"dir_MB"})
    comb = ar_idx.join(mb_idx, how="left").reset_index()
    comb = comb.sort_values("dAUC_AR", ascending=False, na_position="last")

    print(f"\n  {'TF':<11} {'eREG':>5} {'dir_AR':>7} {'dAUC_AR':>9} {'q_AR':>7} "
          f"{'dir_MB':>6} {'dAUC_MB':>9} {'q_MB':>7} "
          f"{'RNA_FC':>8} {'n_tgt':>6} {'lyrs':>5}")
    print("  " + "─" * 88)
    for _, r in comb.iterrows():
        ereg_s  = "✓" if r["in_eRegulon"] else "·"
        dauc_ar = f"{r['dAUC_AR']:+.4f}" if not np.isnan(r["dAUC_AR"]) else "   -    "
        q_ar    = (f"{r['q_AR']:.4f}"
                   if not np.isnan(r.get("q_AR", np.nan)) else "  -    ")
        dauc_mb = (f"{r['dAUC_MB']:+.4f}"
                   if not np.isnan(r.get("dAUC_MB", np.nan)) else "   -    ")
        q_mb    = (f"{r['q_MB']:.4f}"
                   if not np.isnan(r.get("q_MB", np.nan)) else "  -    ")
        rfc     = (f"{r['TF_RNA_log2FC']:+.3f}{'†' if r.get('TF_RNA_sig') else ' '}"
                   if not np.isnan(r["TF_RNA_log2FC"]) else "   -   ")
        nt      = str(int(r["n_ereg_targets"])) if r["n_ereg_targets"] > 0 else "-"
        print(f"  {r['TF']:<11} {ereg_s:>5} {r['dir_AR']:>7} "
              f"{dauc_ar:>9} {q_ar:>7} {r.get('dir_MB',''):>6} "
              f"{dauc_mb:>9} {q_mb:>7} {rfc:>8} {nt:>6} "
              f"{int(r['lyrs_AR']):>5}")
    print(f"\n  ✓=eRegulon  †=RNA sig  dir_AR=act vs rest  dir_MB=MG vs bg  "
          f"lyrs=n evidence layers (AUC+RNA+ATAC)")
    comb.to_csv(OUTDIR / "TF_combined_evidence.csv", index=False)

    print(f"\n  PRIORITY CANDIDATES (sig in BOTH act vs rest AND MG vs bg, q<0.05):")
    priority = comb[(comb["q_AR"] < 0.05) & (comb["q_MB"] < 0.05) & comb["in_eRegulon"]]\
        .sort_values("dAUC_AR", ascending=False)
    if len(priority):
        print(priority[["TF","dir_AR","dAUC_AR","q_AR",
                         "dir_MB","dAUC_MB","q_MB",
                         "TF_RNA_log2FC","n_ereg_targets","lyrs_AR"]]
              .to_string(index=False))
    else:
        print("  None pass both thresholds. Top eRegulon TFs by act vs rest q_AR:")
        print(comb[comb["in_eRegulon"]].nsmallest(5, "q_AR")
              [["TF","dir_AR","dAUC_AR","q_AR","dAUC_MB","q_MB"]].to_string(index=False))
else:
    print("  TF tables not available — skipped")

# ════════════════════════════════════════════════════════════════════
# PART D — Extended TF summary (no eRegulon)
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART D — EXTENDED TF SUMMARY (no eRegulon)")
print("=" * 70)
print("  Ranked by mean tf_to_gene importance across ALL cell types (not MG-specific).")
print("  RNA_FC and ATAC_FC are MG-specific (activated vs resting).")
print()
print(f"  {'TF':<12} {'t2g_n':>7} {'t2g_imp':>9} {'RNA_FC':>8} {'RNA_p':>10} "
      f"{'ATAC_FC':>8} {'in_ereg_tgt':>11}  top_tf2g_targets")
print("  " + "─" * 90)
for tf in extended_tfs:
    sub    = tf2g[tf2g["TF"] == tf] if tf2g is not None else pd.DataFrame()
    rna_r  = da_rna.loc[tf]      if tf in da_rna.index else None
    comb_r = (da_combined.loc[tf]
              if len(da_combined) and tf in da_combined.index else None)
    rfc    = (f"{rna_r['avg_log2FC']:>+8.3f}{'†' if rna_r['p_val_adj']<0.05 else ' '}"
              if rna_r is not None else "       -")
    rp     = (f"{rna_r['p_val_adj']:>10.2e}" if rna_r is not None else "         -")
    atfc   = (f"{comb_r['avg_log2FC.activity']:>+8.3f}"
              if comb_r is not None else "       -")
    in_tgt = "✓" if tf.upper() in ereg_target_set else "·"
    tgt_str = ("|".join(sub.nlargest(5, "importance")["target"].tolist())
               if tf2g is not None and len(sub) else "")
    print(f"  {tf:<12} {len(sub):>7} {sub['importance'].mean():>9.3f} "
          f"{rfc} {rp} {atfc} {in_tgt:>11}  {tgt_str}")

if len(comb):
    print(f"\n  Extended TF significance across both comparisons:")
    ext_rows = comb[~comb["in_eRegulon"]].copy()
    print(f"  {'TF':<12} {'dir_AR':>7} {'RNA_FC_AR':>10} {'RNA_p_AR':>10} {'dir_MB':>7}")
    print("  " + "─" * 55)
    for _, r in ext_rows.iterrows():
        rfc = (f"{r['TF_RNA_log2FC']:>+10.3f}{'†' if r.get('TF_RNA_sig') else ' '}"
               if not np.isnan(r["TF_RNA_log2FC"]) else "         -")
        rp  = (f"{r['TF_RNA_padj']:>10.2e}"
               if not np.isnan(r["TF_RNA_padj"]) else "         -")
        print(f"  {r['TF']:<12} {r['dir_AR']:>7} {rfc} {rp} {r.get('dir_MB',''):>7}")

# ════════════════════════════════════════════════════════════════════
# PART E — Top eRegulon TF regulation audit (data-driven)
#
# Runs a generic regulation mode audit for the TOP_TF_AUDIT most
# activated eRegulon TFs (by ΔAUC), plus flags any TFs where AUC
# direction contradicts RNA direction, and detects feedback loops
# (TF_A → TF_B where TF_B targets openness was pre-established).
# No TF names are hard-coded — everything is derived from the data.
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART E — TOP eRegulon TF REGULATION AUDIT")
print("=" * 70)

if eregs is not None and len(tf_ar):
    # Determine which TFs to audit: top N by |ΔAUC| among reliable eRegulon TFs
    audit_tfs = (tf_ar[tf_ar["in_eRegulon"] & tf_ar["delta_AUC"].notna()]
                 .reindex(tf_ar.index)
                 .sort_values("delta_AUC", ascending=False, key=abs)
                 .head(TOP_TF_AUDIT)["TF"].tolist())
    print(f"  Auditing top {len(audit_tfs)} eRegulon TFs by |ΔAUC|: {audit_tfs}")

    reg_map = {1: "activating", -1: "repressing", 0: "unknown"}

    for tf in audit_tfs:
        print(f"\n  {'─'*60}")
        print(f"  TF: {tf}")
        ereg_tf = eregs[eregs["TF"] == tf]
        print(f"  eRegulon target genes: {ereg_tf['Gene'].nunique()}")

        # Regulation mode breakdown
        if "regulation" in ereg_tf.columns:
            reg_cts  = ereg_tf["regulation"].value_counts().to_dict()
            total    = max(len(ereg_tf), 1)
            act_frac = reg_cts.get(1, 0) / total
            print(f"  Regulation mode (region-gene links):")
            for k, n in sorted(reg_cts.items()):
                print(f"    {reg_map.get(k,k)} ({k}): {n}  ({n/total:.1%})")
            if act_frac > 0.6:
                print(f"  → {tf} acts predominantly as ACTIVATOR")
            elif act_frac < 0.4:
                print(f"  → {tf} acts predominantly as REPRESSOR "
                      f"(elevated AUC in activated MG may reflect de-repression)")
            else:
                print(f"  → Mixed — context-dependent per region")

        # Target RNA direction
        tgts_in = [g for g in ereg_tf["Gene"].unique() if g in da_rna.index]
        if tgts_in:
            sub_rna = da_rna.loc[tgts_in, ["avg_log2FC","p_val_adj"]]
            n_up    = ((sub_rna["avg_log2FC"] >  DEG_FC_THRESH) &
                       (sub_rna["p_val_adj"] < 0.05)).sum()
            n_dn    = ((sub_rna["avg_log2FC"] < -DEG_FC_THRESH) &
                       (sub_rna["p_val_adj"] < 0.05)).sum()
            print(f"  Target RNA direction (FC>{DEG_FC_THRESH}, padj<0.05):  "
                  f"up={n_up}  down={n_dn}  mean_FC={sub_rna['avg_log2FC'].mean():+.3f}")
            if n_up > n_dn * 2:
                print(f"  → Targets UP → {tf} is an ACTIVATOR in activated MG")
            elif n_dn > n_up * 2:
                print(f"  → Targets DOWN → {tf} is a REPRESSOR (resting-state driver)")
            else:
                print(f"  → Mixed — examine individual targets")

        # tf_to_gene rho
        if tf2g is not None:
            t2g_tf = tf2g[tf2g["TF"] == tf]
            if len(t2g_tf):
                pos_rho = (t2g_tf["rho"] > 0).sum()
                print(f"  tf_to_gene: {len(t2g_tf)} links  "
                      f"pos_rho={pos_rho/len(t2g_tf):.1%}  "
                      f"mean_rho={t2g_tf['rho'].mean():+.3f}")

        # Feedback loop detection: does this TF regulate another audited TF?
        for other_tf in audit_tfs:
            if other_tf == tf:
                continue
            link_ereg = eregs[(eregs["TF"] == tf) & (eregs["Gene"] == other_tf)]
            link_t2g  = (tf2g[(tf2g["TF"] == tf) & (tf2g["target"] == other_tf)]
                         if tf2g is not None else pd.DataFrame())
            if len(link_ereg) or len(link_t2g):
                tf_row    = tf_ar[tf_ar["TF"] == tf]
                other_row = tf_ar[tf_ar["TF"] == other_tf]
                tf_dir    = tf_row["direction"].values[0]    if len(tf_row)    else "?"
                other_dir = other_row["direction"].values[0] if len(other_row) else "?"
                print(f"  → Potential feedback: {tf} ({tf_dir}) regulates {other_tf} ({other_dir})")
                if len(link_ereg):
                    cols = [c for c in ["TF","Gene","importance_TF2G","regulation","triplet_rank"]
                            if c in link_ereg.columns]
                    print(f"    eRegulon: {link_ereg[cols].to_string(index=False)}")
                if len(link_t2g):
                    print(f"    tf_to_gene: importance={link_t2g['importance'].values[0]:.3f}  "
                          f"rho={link_t2g['rho'].values[0]:.3f}")

    # Paradox flags: meaningful AUC/RNA direction contradiction
    print(f"\n  PARADOX FLAGS — AUC direction contradicts RNA direction (|dAUC|>0.001, RNA sig):")
    print(f"  {'─'*65}")
    any_paradox = False
    for _, r in tf_ar[tf_ar["in_eRegulon"]].iterrows():
        if np.isnan(r["delta_AUC"]) or np.isnan(r["TF_RNA_log2FC"]):
            continue
        if abs(r["delta_AUC"]) < 0.001:
            continue
        auc_dir = "ACT" if r["delta_AUC"] > 0 else "REST"
        rna_dir = "ACT" if r["TF_RNA_log2FC"] > 0 else "REST"
        if auc_dir != rna_dir and r["TF_RNA_sig"]:
            any_paradox = True
            print(f"  {r['TF']:<10} AUC→{auc_dir}  RNA→{rna_dir}  "
                  f"dAUC={r['delta_AUC']:+.4f}  RNA_FC={r['TF_RNA_log2FC']:+.3f}")
            print(f"           Possible explanations: post-transcriptional regulation, "
                  f"chromatin pre-positioning, or cofactor availability. "
                  f"Verify with protein-level data.")
    if not any_paradox:
        print("  None detected.")
else:
    print("  eRegulons not loaded or TF stats not available.")

# ════════════════════════════════════════════════════════════════════
# PART F — Top pathway genes (data-driven)
#
# Instead of hard-coded pathway lists, auto-derives top regulated
# genes by FC and annotates each with eRegulon TFs, tf_to_gene
# regulators, and JASPAR motif p-values. Optionally extends with
# any user-provided pathway gene sets stored in external CSVs.
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART F — TOP PATHWAY GENES (data-driven)")
print("RNA + ATAC + eRegulon + JASPAR motif evidence")
print("=" * 70)
print(f"  Showing top {TOP_PATHWAY_GENES} activated and top {TOP_PATHWAY_GENES} repressed genes.")
print(f"  For curated pathway gene sets, add CSV files to {BASE} with columns: gene, pathway")

def print_pathway_gene_block(gene_list, gt, block_label):
    """Print RNA/ATAC/eRegulon/JASPAR stats for a list of genes."""
    hits     = [(g, gt.loc[g] if g in gt.index else None) for g in gene_list]
    has_data = [(g, r) for g, r in hits if r is not None]
    if not has_data:
        print(f"  [{block_label}] — none found in DEG table")
        return
    print(f"\n  [{block_label}]")
    print(f"  {'Gene':<14} {'RNA_FC':>8} {'RNA_p':>10} {'ATAC_FC':>8} "
          f"{'Dir':>6} {'eReg_TFs':<22}  top_tf2g  JASPAR_p")
    print("  " + "─" * 95)
    for gene, r in has_data:
        rfc  = f"{r['RNA_log2FC']:>+8.3f}" if not np.isnan(r["RNA_log2FC"]) else "       -"
        rp   = f"{r['RNA_padj']:>10.2e}"   if not np.isnan(r["RNA_padj"])   else "         -"
        atfc = (f"{r['ATAC_log2FC']:>+8.3f}"
                if "ATAC_log2FC" in r.index and not np.isnan(r["ATAC_log2FC"])
                else "       -")
        sig  = "★" if not np.isnan(r.get("RNA_padj", np.nan)) and r["RNA_padj"] < 0.05 else " "
        d    = ("ACT↑" if r["RNA_padj"] < 0.05 and r["RNA_log2FC"] > 0 else
                "REST↑" if r["RNA_padj"] < 0.05 else "n.s.")
        ereg = r["eReg_TFs"][:22] if r.get("eReg_TFs") else ""
        print(f"  {gene:<14}{sig}{rfc} {rp} {atfc} "
              f"{d:>6} {ereg:<22}  {_top_t2g(gene)}{_jasp_p(gene)}")
    missing = [g for g, r in hits if r is None]
    if missing:
        print(f"  Not in DEG data: {missing}")

# Auto-derived: top activated genes
top_act_genes = (gt_ar[gt_ar["RNA_sig"] & (gt_ar["RNA_log2FC"] > 0)]
                 .sort_values(["n_sig","RNA_log2FC"], ascending=[False, False])
                 .head(TOP_PATHWAY_GENES).index.tolist())
print_pathway_gene_block(top_act_genes, gt_ar, f"Top {TOP_PATHWAY_GENES} activated genes by FC+sig")

# Auto-derived: top repressed genes
top_rest_genes = (gt_ar[gt_ar["RNA_sig"] & (gt_ar["RNA_log2FC"] < 0)]
                  .sort_values(["n_sig","RNA_log2FC"], ascending=[False, True])
                  .head(TOP_PATHWAY_GENES).index.tolist())
print_pathway_gene_block(top_rest_genes, gt_ar, f"Top {TOP_PATHWAY_GENES} repressed genes by FC+sig")

# Auto-derived: top eRegulon targets (most TF regulators, regardless of direction)
top_ereg_tgts = (pd.Series(list(ereg_target_set))
                 .map(lambda g: eregs[eregs["Gene"].str.upper() == g]["TF"].nunique()
                      if eregs is not None else 0)
                 .sort_values(ascending=False)
                 .head(TOP_PATHWAY_GENES).index.tolist()
                 if eregs is not None else [])
# Recover original case from avg expression index or da_rna
gene_case_map = {g.upper(): g for g in da_rna.index}
top_ereg_genes = [gene_case_map.get(g, g) for g in top_ereg_tgts]
if top_ereg_genes:
    print_pathway_gene_block(top_ereg_genes, gt_ar,
                             f"Top {TOP_PATHWAY_GENES} eRegulon targets (most TF regulators)")

# Optional: user-provided pathway gene sets (any CSV with 'gene' and 'pathway' columns)
for csv_path in sorted(BASE.glob("pathway_genes*.csv")):
    try:
        pw_df = pd.read_csv(csv_path)
        if "gene" in pw_df.columns and "pathway" in pw_df.columns:
            for pw, grp in pw_df.groupby("pathway"):
                print_pathway_gene_block(grp["gene"].tolist(), gt_ar, pw)
        print(f"  Loaded pathway file: {csv_path.name}")
    except Exception as e:
        print(f"  Could not load {csv_path.name}: {e}")

# TFs regulating top TF→gene hubs (who drives the drivers?)
print(f"\n  TF regulators of top activated genes:")
print(f"  {'─'*65}")
for gene in top_act_genes[:10]:
    t2g_str = _top_t2g(gene)
    jasp    = _jasp_p(gene)
    if t2g_str or gene in ereg_target_map:
        print(f"    {gene:<14} tf2g: {t2g_str:<30} eReg: {ereg_target_map.get(gene,'')[:20]}"
              f"{jasp}")

# Save combined pathway table
pathway_rows = []
for gene in set(top_act_genes + top_rest_genes + top_ereg_genes):
    if gene not in gt_ar.index:
        continue
    r        = gt_ar.loc[gene]
    t2g_hits = (tf2g[tf2g["target"].str.upper() == gene.upper()]
                if tf2g is not None else pd.DataFrame())
    jasp_p = jasp_fc = np.nan
    if jaspar_rec is not None:
        jrow = jaspar_rec[jaspar_rec["motif.name"].str.upper() == gene.upper()]
        if len(jrow):
            jasp_p  = jrow.iloc[0]["pvalue"]
            jasp_fc = jrow.iloc[0].get("motif.fc", np.nan)
    pathway_rows.append(dict(
        gene=gene,
        RNA_log2FC=r["RNA_log2FC"], RNA_padj=r["RNA_padj"], RNA_sig=r["RNA_sig"],
        ATAC_log2FC=r.get("ATAC_log2FC", np.nan), ATAC_sig=r.get("ATAC_sig", False),
        concordant=r.get("concordant", False),
        in_eReg_target=r["in_eReg_target"], eReg_TFs=r["eReg_TFs"],
        top_tf2g=("|".join(t2g_hits.nlargest(3, "importance")["TF"].tolist())
                  if len(t2g_hits) else ""),
        JASPAR_pvalue=jasp_p, JASPAR_FC=jasp_fc,
    ))
if pathway_rows:
    pd.DataFrame(pathway_rows).to_csv(OUTDIR / "top_pathway_genes.csv", index=False)

# ════════════════════════════════════════════════════════════════════
# PART G — TF→gene network + hub genes
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART G — TF→GENE NETWORK + HUB GENES")
print("=" * 70)

act_sig_tfs  = (comb[(comb["dAUC_AR"] > 0) & (comb["q_AR"] < 0.05) &
                      comb["in_eRegulon"]]["TF"].tolist()
                if len(comb) else [])
rest_sig_tfs = (comb[(comb["dAUC_AR"] < 0) & (comb["q_AR"] < 0.05) &
                      comb["in_eRegulon"]]["TF"].tolist()
                if len(comb) else [])
all_net_tfs  = list(set(reliable_tfs + extended_tfs))

G = nx.DiGraph()
for tf in all_net_tfs:
    state   = ("activated" if tf in act_sig_tfs else
               "resting"   if tf in rest_sig_tfs else "neutral")
    t2g_sub = tf2g[tf2g["TF"] == tf] if tf2g is not None else pd.DataFrame()
    thresh  = np.percentile(t2g_sub["importance"], IMP_PCT) if len(t2g_sub) else 0
    if eregs_rel is not None:
        for _, row in eregs_rel[eregs_rel["TF"] == tf].iterrows():
            G.add_edge(tf, row["Gene"], source="eRegulon",
                       weight=float(row.get("importance_TF2G", 1)),
                       regulation=int(row.get("regulation", 0)))
    for _, row in t2g_sub[t2g_sub["importance"] >= thresh].iterrows():
        if not G.has_edge(tf, row["target"]):
            G.add_edge(tf, row["target"], source="tf2g",
                       weight=float(row["importance"]), regulation=0)
    G.nodes[tf]["node_type"] = "TF"
    G.nodes[tf]["MG_state"]  = state

for n, d in G.nodes(data=True):
    if d.get("node_type") != "TF":
        G.nodes[n]["node_type"] = "gene"
        rr = da_rna.loc[n] if n in da_rna.index else None
        G.nodes[n]["RNA_log2FC"] = float(rr["avg_log2FC"])      if rr is not None else 0.0
        G.nodes[n]["RNA_sig"]    = bool(rr["p_val_adj"] < 0.05) if rr is not None else False

print(f"\n  Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"  Activated TFs (q<0.05): {act_sig_tfs}")
print(f"  Resting TFs  (q<0.05): {rest_sig_tfs}")

hub_rows = []
for gene, d in G.nodes(data=True):
    if d.get("node_type") != "gene":
        continue
    preds  = list(G.predecessors(gene))
    if len(preds) < 2:
        continue
    act_r  = [t for t in preds if t in act_sig_tfs]
    rest_r = [t for t in preds if t in rest_sig_tfs]
    rr     = da_rna.loc[gene] if gene in da_rna.index else None
    hub_rows.append(dict(
        gene=gene, n_regulators=len(preds),
        act_TFs="|".join(act_r), rest_TFs="|".join(rest_r),
        all_TFs="|".join(sorted(preds)),
        is_transition=bool(act_r and rest_r),
        RNA_log2FC=round(float(rr["avg_log2FC"]), 3) if rr is not None else np.nan,
        RNA_padj=round(float(rr["p_val_adj"]), 6)    if rr is not None else np.nan,
        RNA_sig=bool(rr["p_val_adj"] < 0.05)         if rr is not None else False,
        in_eRegulon=gene in ereg_target_set,
    ))

hub_df   = (pd.DataFrame(hub_rows).sort_values("n_regulators", ascending=False)
            if hub_rows else pd.DataFrame())
trans_df = (hub_df[hub_df["is_transition"]].sort_values("RNA_log2FC", ascending=False)
            if len(hub_df) else pd.DataFrame())

print(f"\n  Hub genes (≥2 TF regulators): {len(hub_df)}")
print(f"  Transition genes (regulated by both act and rest TFs): {len(trans_df)}")
if len(hub_df):
    print(f"\n  Top 15 hub genes:")
    print(hub_df[["gene","n_regulators","act_TFs","rest_TFs",
                  "RNA_log2FC","RNA_sig"]].head(15).to_string(index=False))
if len(trans_df):
    print(f"\n  Top transition genes:")
    print(trans_df[["gene","act_TFs","rest_TFs",
                    "RNA_log2FC","RNA_padj"]].head(10).to_string(index=False))

nx.write_graphml(G, str(OUTDIR / "MG_network.graphml"))
if len(hub_df):
    hub_df.to_csv(OUTDIR / "hub_genes.csv", index=False)
if len(trans_df):
    trans_df.to_csv(OUTDIR / "transition_genes.csv", index=False)
if eregs_rel is not None and len(eregs_rel):
    tgt_rows = []
    for _, row in eregs_rel.iterrows():
        rr = da_rna.loc[row["Gene"]] if row["Gene"] in da_rna.index else None
        tgt_rows.append(dict(
            Gene=row["Gene"], TF=row["TF"], Region=row["Region"],
            importance_TF2G=row.get("importance_TF2G", np.nan),
            importance_R2G=row.get("importance_R2G",   np.nan),
            regulation=row.get("regulation", np.nan),
            triplet_rank=row.get("triplet_rank", np.nan),
            Gene_RNA_log2FC=float(rr["avg_log2FC"]) if rr is not None else np.nan,
            Gene_RNA_padj=float(rr["p_val_adj"])    if rr is not None else np.nan,
        ))
    pd.DataFrame(tgt_rows).to_csv(OUTDIR / "target_gene_TF_links.csv", index=False)

# ════════════════════════════════════════════════════════════════════
# PART H — JASPAR motif enrichment summary (data-driven)
#
# Reports all significant enriched motifs from both JASPAR databases.
# Family-level grouping is derived dynamically from the motif names
# present in the enrichment results — no TF families are hard-coded.
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART H — JASPAR MOTIF ENRICHMENT SUMMARY")
print("Human-only JASPAR vs SCENIC+-reconciled PWMs")
print("=" * 70)

if jaspar_rec is not None:
    sig_rec = jaspar_rec[jaspar_rec["pvalue"] < 0.05]
    in_db   = (sig_rec[sig_rec["in_scenic_db"].astype(str).str.lower() == "true"]
               if "in_scenic_db" in sig_rec.columns else sig_rec)
    print(f"\n  SCENIC+-reconciled: {len(jaspar_rec)} motifs tested  "
          f"|  sig (p<0.05): {len(sig_rec)}  |  in SCENIC+ DB: {len(in_db)}")
    cols_r = [c for c in ["motif.name","scenic_TF","pvalue","motif.fc",
                            "fold.enrichment","in_scenic_db"]
              if c in jaspar_rec.columns]

    # Top 20 enriched overall
    print(f"\n  Top 20 enriched (SCENIC+-reconciled, p<0.05):")
    print(sig_rec[cols_r].head(20).to_string(index=False))

    # Data-driven TF family grouping: extract common prefixes from motif names
    # (e.g. "ELK", "SOX", "KLF", "SP", "EGR" are auto-detected from the results)
    if "motif.name" in jaspar_rec.columns:
        import re
        motif_names  = sig_rec["motif.name"].dropna().str.upper().tolist()
        # Extract 2–6 char alphabetic prefixes that appear in ≥2 motifs
        prefix_counts = {}
        for name in motif_names:
            m = re.match(r"^([A-Z]{2,6})", name)
            if m:
                pfx = m.group(1)
                prefix_counts[pfx] = prefix_counts.get(pfx, 0) + 1
        # Show any family with ≥2 significant hits
        active_families = {k: v for k, v in prefix_counts.items() if v >= 2}
        if active_families:
            print(f"\n  TF families with ≥2 significant motifs (auto-detected):")
            for family, count in sorted(active_families.items(),
                                        key=lambda x: -x[1]):
                hits = sig_rec[sig_rec["motif.name"].str.upper()
                               .str.startswith(family, na=False)]
                print(f"\n  [{family}* family — {count} significant motifs]")
                print(hits[cols_r].sort_values("pvalue").to_string(index=False))

    # Cross-database comparison
    if jaspar_old is not None:
        sig_old = jaspar_old[jaspar_old["pvalue"] < 0.05]
        print(f"\n  Human-only JASPAR: {len(jaspar_old)} tested  |  sig: {len(sig_old)}")

    if jaspar_comp is not None:
        cols_c = [c for c in ["motif.name","pvalue.human_jaspar",
                               "pvalue.scenic_reconciled","scenic_TF","in_scenic_db"]
                  if c in jaspar_comp.columns]
        both_sig = jaspar_comp[
            jaspar_comp["pvalue.human_jaspar"].notna() &
            (jaspar_comp["pvalue.human_jaspar"] < 0.05) &
            jaspar_comp["pvalue.scenic_reconciled"].notna() &
            (jaspar_comp["pvalue.scenic_reconciled"] < 0.05)]
        new_only = jaspar_comp[
            (jaspar_comp["pvalue.human_jaspar"].isna() |
             (jaspar_comp["pvalue.human_jaspar"] >= 0.05)) &
            jaspar_comp["pvalue.scenic_reconciled"].notna() &
            (jaspar_comp["pvalue.scenic_reconciled"] < 0.05)]
        dropped = jaspar_comp[
            jaspar_comp["pvalue.human_jaspar"].notna() &
            (jaspar_comp["pvalue.human_jaspar"] < 0.05) &
            (jaspar_comp["pvalue.scenic_reconciled"].isna() |
             (jaspar_comp["pvalue.scenic_reconciled"] >= 0.05))]
        print(f"\n  Comparison (p<0.05):")
        print(f"    Significant in BOTH:          {len(both_sig)}")
        print(f"    New (reconciled only):         {len(new_only)}")
        print(f"    Dropped (human-JASPAR only):   {len(dropped)}")
        for label, df_sub in [("Both", both_sig), ("New (reconciled only)", new_only),
                               ("Dropped (human-JASPAR only)", dropped)]:
            if len(df_sub):
                sort_col = ("pvalue.scenic_reconciled"
                            if "pvalue.scenic_reconciled" in df_sub.columns
                            else "pvalue.human_jaspar")
                print(f"\n  Top 10 — {label}:")
                print(df_sub.sort_values(sort_col)[cols_c].head(10).to_string(index=False))
else:
    print("  enriched_motifs_scenic_reconciled.csv not found — "
          "run signac_scenic_motif_enrichment.R first")

# ════════════════════════════════════════════════════════════════════
# PART I — Output file summary
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART I — OUTPUT FILES")
print("=" * 70)
for f in sorted(OUTDIR.iterdir()):
    print(f"  {f.name:<50} {f.stat().st_size/1024:>8.1f} KB")

print("\n" + "=" * 70)
print("COMPLETE")
print("=" * 70)

SCENIC+ DIAGNOSTIC + MG ANALYSIS

──────────────────────────────────────────────────────────────────────
  1. PIPELINE FILES
──────────────────────────────────────────────────────────────────────
  ✓ cisTopic pkl                                     957.6 MB
  ✓ RNA h5ad                                         203.9 MB
  ✓ motifs.filtered.tbl                               38.4 MB
  ✓ scores feather (DEM)                             823.5 MB
  ✓ rankings feather (CTX)                           823.5 MB
  ✓ region_sets folder                                 0.0 MB
  ✓ ACC_GEX metacell h5mu                           1943.4 MB
  ✓ dem_results.hdf5                                   7.9 MB
  ✓ ctx_results.hdf5                                   6.4 MB
  ✓ cistromes_extended.h5ad                            1.0 MB
  ✓ cistromes_direct.h5ad                              0.0 MB
  ✓ tf_to_gene_adj.tsv                                80.1 MB
  ✓ region_to_gene_adj.tsv                            65.4 M

In [9]:
# ================================================================
# CELL 19 — ACTIVATED vs RESTING MG: COMPLETE COMPARISON
#
# Merges the RNA batch-control specificity analysis (former cell 19)
# with a full multiome deep-dive across all SCENIC+ output layers.
#
# Requires cell 18 to have run (da_rna, da_combined, eregs, eregs_rel,
# ereg_sizes, reliable_tfs, ereg_target_set, ereg_target_map, tf2g,
# r2g, auc_m, act_auc, rest_auc, comb, new_obs, scplus_mdata,
# ACT_LABEL, REST_LABEL, MG_LABELS, DEG_FC_THRESH, OUTDIR in scope).
#
# Structure:
#   BLOCK 1 — RNA: top activated-MG genes + batch confounding assessment
#   BLOCK 2 — RNA: data-derived gene groups + resting MG identity check
#   BLOCK 3 — RNA: per-cell-type expression (MG-specificity confirmation)
#   BLOCK 4 — eRegulon AUC: per-state activity scores + delta ranking
#   BLOCK 5 — eRegulon targets: state-specific sets + concordance
#   BLOCK 6 — region_to_gene: MG regulatory links filtered to DE genes
#   BLOCK 7 — Cross-layer TF evidence summary (RNA + ATAC + AUC + r2g)
#   BLOCK 8 — Save all outputs + interpretation summary
# ================================================================
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# ── Config (edit here) ────────────────────────────────────────────
BASE        = Path("/mnt/c/Users/jacwe/Downloads/SAMAP/")
avg_path    = BASE / "celltype_mean_expression.csv"
da_rna_path = BASE / "da_rna_full.csv"

PSEUDOCOUNT  = 0.01   # prevents division-by-zero for unexpressed genes
RATIO_STRONG = 3.0    # ratio > this → MG-specific
RATIO_ENRICH = 1.5    # ratio between ENRICH and STRONG → MG-enriched
MIN_EXPR     = 0.05   # min activated-MG expression to classify
TOP_ACT      = 40     # top N activated-MG genes by FC (Block 1)
TOP_REST     = 20     # top N resting-MG genes by FC  (Block 2)
TOP_SPECIFIC = 15     # top N MG-specific genes for per-cell-type table (Block 3)
TOP_AUTO_GRP = 20     # genes per auto-derived group (Block 2)
TOP_DELTA    = 20     # top N eRegulons by |delta AUC| to show (Block 4)
MIN_R2G_RHO  = 0.1   # min |rho| for region→gene links shown (Block 6)

print("=" * 70)
print("ACTIVATED vs RESTING MG — COMPLETE COMPARISON")
print("=" * 70)
print(f"  {ACT_LABEL} (NM) vs {REST_LABEL} (SS)")
print("  ⚠ Complete sample confounding: states are 100% separated by sample.")
print("  Blocks 1–3 use mean_non_MG as a sample-balanced control to assess")
print("  whether signals are MG cell-state vs NM treatment-wide effects.")
print("  Blocks 4–7 use the multiome layers to corroborate RNA findings.")

# ── Guard: load avg expression if not already done ────────────────
if not avg_path.exists():
    print(f"\n  ERROR: {avg_path} not found. Run in R:")
    print("    avg <- AverageExpression(rna, group.by='celltype',")
    print("                             assay='RNA', layer='data')$RNA")
    print("    write.csv(avg, 'celltype_mean_expression.csv')")
    raise SystemExit

avg = pd.read_csv(avg_path, index_col=0)
avg.columns = [c.replace(".", " ").strip() for c in avg.columns]

act_col  = next((c for c in avg.columns
                 if "activated" in c.lower() and "mg" in c.lower()), None)
rest_col = next((c for c in avg.columns
                 if "resting"   in c.lower() and "mg" in c.lower()), None)
if act_col is None or rest_col is None:
    print(f"  ERROR: Could not find MG columns in: {avg.columns.tolist()}")
    raise SystemExit

non_mg_cols        = [c for c in avg.columns
                      if "mg" not in c.lower() and "muller" not in c.lower()]
avg["mean_non_mg"] = avg[non_mg_cols].mean(axis=1)

# Load da_rna if not in scope from cell 18
try:
    da_rna
except NameError:
    da_rna = pd.read_csv(da_rna_path).set_index("gene")

# ════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ════════════════════════════════════════════════════════════════════

def classify(act_m, non_mg):
    ratio = act_m / (non_mg + PSEUDOCOUNT)
    if act_m < MIN_EXPR:
        return ratio, "low expr"
    elif ratio >= RATIO_STRONG:
        return ratio, "✓ MG-specific"
    elif ratio >= RATIO_ENRICH:
        return ratio, "~ MG-enriched"
    else:
        return ratio, "⚠ NM-broad"

def get_row(gene):
    in_avg = gene in avg.index
    in_rna = gene in da_rna.index
    if not in_avg and not in_rna:
        return None
    act_m  = avg.loc[gene, act_col]       if in_avg else np.nan
    rest_m = avg.loc[gene, rest_col]      if in_avg else np.nan
    non_mg = avg.loc[gene, "mean_non_mg"] if in_avg else np.nan
    fc     = da_rna.loc[gene, "avg_log2FC"] if in_rna else np.nan
    padj   = da_rna.loc[gene, "p_val_adj"]  if in_rna else np.nan
    return fc, padj, act_m, rest_m, non_mg

def fmt(val, fmt_str, fallback="    n/a"):
    return format(val, fmt_str) if not (isinstance(val, float) and np.isnan(val)) else fallback

def print_gene_row(gene, fc, padj, act_m, rest_m, non_mg, show_padj=False):
    ratio, verdict = classify(act_m, non_mg)
    ratio_s = (fmt(ratio, "6.1f") + "x") if not np.isnan(ratio) else "    n/a"
    if show_padj:
        print(f"  {gene:<26} {fmt(fc,'+7.2f')} {fmt(padj,'.1e','      n/a'):>10} "
              f"{fmt(act_m,'7.3f')} {fmt(rest_m,'7.3f')} "
              f"{fmt(non_mg,'9.3f','      n/a')} {ratio_s}  {verdict}")
    else:
        print(f"  {gene:<26} {fmt(fc,'+7.2f')} "
              f"{fmt(act_m,'7.3f')} {fmt(rest_m,'7.3f')} "
              f"{fmt(non_mg,'9.3f','      n/a')} {ratio_s}  {verdict}")
    return ratio, verdict

# ════════════════════════════════════════════════════════════════════
# BLOCK 1 — RNA: Top activated-MG genes + batch confounding
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 1: TOP {TOP_ACT} ACTIVATED-MG GENES BY RNA FC")
print(f"  (ratio = act_MG_mean / mean_non_MG — controls for NM sample effect)")
print(f"{'─'*70}")
print(f"\n  {'Gene':<26} {'FC':>7} {'RNA_padj':>10} {'act_MG':>7} "
      f"{'rest_MG':>7} {'non_MG_avg':>9} {'ratio':>7}  verdict")
print("  " + "─" * 85)

top_act_rna = (da_rna[da_rna["avg_log2FC"] > 0]
               .sort_values("avg_log2FC", ascending=False)
               .head(TOP_ACT))

counts   = {"✓ MG-specific": [], "~ MG-enriched": [], "⚠ NM-broad": [], "low expr": []}
rows_csv = []

for gene in top_act_rna.index:
    row = get_row(gene)
    if row is None:
        print(f"  {gene:<26}  not in expression matrix")
        continue
    fc, padj, act_m, rest_m, non_mg = row
    ratio, verdict = print_gene_row(gene, fc, padj, act_m, rest_m, non_mg, show_padj=True)
    counts[verdict].append(gene)
    rows_csv.append(dict(gene=gene, RNA_log2FC=fc, RNA_padj=padj,
                         act_MG_mean=act_m, rest_MG_mean=rest_m,
                         non_MG_mean=non_mg, ratio=ratio, verdict=verdict))

print(f"\n  SUMMARY (top {TOP_ACT} activated-MG genes):")
for v, genes in counts.items():
    print(f"    {v:<30} {len(genes):>3}  {genes[:8]}")

n_total    = sum(len(v) for v in counts.values())
broad_frac = len(counts["⚠ NM-broad"]) / max(n_total, 1)
print(f"\n  BATCH CONFOUNDING ASSESSMENT:")
if broad_frac < 0.15:
    print(f"  GOOD: {broad_frac:.0%} NM-broad. Signature is predominantly MG cell-state.")
elif broad_frac < 0.35:
    print(f"  MIXED: {broad_frac:.0%} NM-broad. Focus on MG-specific/enriched genes.")
else:
    print(f"  CONCERN: {broad_frac:.0%} NM-broad. Significant batch confounding.")

# ════════════════════════════════════════════════════════════════════
# BLOCK 2 — RNA: Data-derived gene groups + resting MG identity check
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 2: DATA-DERIVED GENE GROUPS + RESTING MG IDENTITY")
print(f"{'─'*70}")

common_genes = avg.index.intersection(da_rna.index)
summary = pd.DataFrame({
    "act_MG":   avg.loc[common_genes, act_col],
    "rest_MG":  avg.loc[common_genes, rest_col],
    "non_MG":   avg.loc[common_genes, "mean_non_mg"],
    "RNA_FC":   da_rna.loc[common_genes, "avg_log2FC"],
    "RNA_padj": da_rna.loc[common_genes, "p_val_adj"],
})
summary["ratio_act"]  = summary["act_MG"]  / (summary["non_MG"] + PSEUDOCOUNT)
summary["ratio_rest"] = summary["rest_MG"] / (summary["non_MG"] + PSEUDOCOUNT)
summary["verdict"]    = summary.apply(lambda r: classify(r["act_MG"], r["non_MG"])[1], axis=1)
summary["RNA_sig"]    = summary["RNA_padj"] < 0.05

auto_groups = {}

# Top eRegulon targets (most TF regulators)
try:
    ereg_top = (eregs["Gene"].value_counts().head(TOP_AUTO_GRP).index.tolist())
    auto_groups[f"Top eRegulon targets (most TF regulators)"] = ereg_top
except NameError:
    pass

# MG-specific activated genes
mg_spec = (summary[(summary["verdict"] == "✓ MG-specific") &
                    summary["RNA_sig"] & (summary["RNA_FC"] > 0)]
           .sort_values("ratio_act", ascending=False).head(TOP_AUTO_GRP).index.tolist())
if mg_spec:
    auto_groups[f"MG-specific activated (ratio>{RATIO_STRONG}x, sig)"] = mg_spec

# MG-enriched activated genes
mg_enrich = (summary[(summary["verdict"] == "~ MG-enriched") &
                      summary["RNA_sig"] & (summary["RNA_FC"] > 0)]
             .sort_values("ratio_act", ascending=False).head(TOP_AUTO_GRP).index.tolist())
if mg_enrich:
    auto_groups[f"MG-enriched activated ({RATIO_ENRICH}–{RATIO_STRONG}x, sig)"] = mg_enrich

# Resting MG identity genes
rest_id = (summary[(summary["ratio_rest"] > RATIO_STRONG) &
                    summary["RNA_sig"] & (summary["RNA_FC"] < 0)]
           .sort_values("ratio_rest", ascending=False).head(TOP_AUTO_GRP).index.tolist())
if rest_id:
    auto_groups["Resting MG identity (high rest/non-MG, sig, FC<0)"] = rest_id

# NM-broad batch concern
nm_broad = (summary[(summary["verdict"] == "⚠ NM-broad") &
                     summary["RNA_sig"] & (summary["RNA_FC"] > 0)]
            .sort_values("RNA_FC", ascending=False).head(TOP_AUTO_GRP).index.tolist())
if nm_broad:
    auto_groups["NM-broad (high FC, low MG-specificity — batch concern)"] = nm_broad

for grp_label, genes in auto_groups.items():
    print(f"\n  [{grp_label}]")
    print(f"  {'Gene':<22} {'FC':>7} {'act_MG':>7} {'rest_MG':>7} "
          f"{'non_MG':>9} {'ratio':>7}  verdict")
    print("  " + "─" * 72)
    for gene in genes:
        row = get_row(gene)
        if row is None:
            print(f"  {gene:<22}  not in matrix or DEG table")
            continue
        fc, padj, act_m, rest_m, non_mg = row
        ratio, verdict = print_gene_row(gene, fc, padj, act_m, rest_m, non_mg)
        rows_csv.append(dict(gene=gene, RNA_log2FC=fc, RNA_padj=padj,
                             act_MG_mean=act_m, rest_MG_mean=rest_m,
                             non_MG_mean=non_mg, ratio=ratio, verdict=verdict))

# Resting MG genes specificity check
print(f"\n  TOP {TOP_REST} RESTING-MG GENES — SS-SPECIFICITY CHECK")
print(f"  (High rest/non-MG ratio → genuine resting identity; low → SS sample effect)")
print(f"\n  {'Gene':<22} {'FC':>7} {'rest_MG':>7} {'act_MG':>7} "
      f"{'non_MG':>9} {'rest_ratio':>11}  verdict")
print("  " + "─" * 72)

top_rest_rna = (da_rna[da_rna["avg_log2FC"] < 0]
                .sort_values("avg_log2FC").head(TOP_REST))
for gene in top_rest_rna.index:
    row = get_row(gene)
    if row is None:
        continue
    fc, padj, act_m, rest_m, non_mg = row
    rest_ratio = rest_m / (non_mg + PSEUDOCOUNT) if not np.isnan(rest_m) else np.nan
    if np.isnan(rest_ratio):
        verdict_r = "not in matrix"
    elif rest_m < MIN_EXPR:
        verdict_r = "low expr"
    elif rest_ratio >= RATIO_STRONG:
        verdict_r = "✓ MG-identity"
    elif rest_ratio >= RATIO_ENRICH:
        verdict_r = "~ MG-enriched"
    else:
        verdict_r = "⚠ SS-broad"
    ratio_s = (fmt(rest_ratio, "11.1f") + "x") if not np.isnan(rest_ratio) else "          n/a"
    print(f"  {gene:<22} {fmt(fc,'+7.2f')} {fmt(rest_m,'7.3f')} "
          f"{fmt(act_m,'7.3f')} {fmt(non_mg,'9.3f','      n/a')} "
          f"{ratio_s}  {verdict_r}")
    rows_csv.append(dict(gene=gene, RNA_log2FC=fc, RNA_padj=padj,
                         act_MG_mean=act_m, rest_MG_mean=rest_m,
                         non_MG_mean=non_mg, ratio=rest_ratio, verdict=verdict_r))

# ════════════════════════════════════════════════════════════════════
# BLOCK 3 — RNA: Per-cell-type expression for top MG-specific genes
# Confirms signal isn't leaking into other cell types
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 3: PER-CELL-TYPE EXPRESSION — TOP {TOP_SPECIFIC} MG-SPECIFIC GENES")
print(f"  (Signal should be high only in MG columns)")
print(f"{'─'*70}")

top_mg_specific = [g for g in counts["✓ MG-specific"] if g in avg.index][:TOP_SPECIFIC]
if top_mg_specific:
    display_cols = [act_col, rest_col] + non_mg_cols
    available    = [c for c in display_cols if c in avg.columns]
    sub          = avg.loc[top_mg_specific, available].round(3)
    sub.columns  = [c[:12] for c in sub.columns]
    print(f"\n{sub.to_string()}")
else:
    print("  No MG-specific genes found in expression matrix for this set.")

# ════════════════════════════════════════════════════════════════════
# BLOCK 4 — eRegulon AUC: per-state activity + delta ranking
#
# For each eRegulon, computes mean AUC in activated vs resting MG
# metacells and ranks by delta. This is the chromatin accessibility
# dimension of TF activity — independent of RNA expression.
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 4: eRegulon AUC — PER-STATE ACTIVITY")
print(f"  (AUC = fraction of eRegulon target regions in top-accessible regions)")
print(f"  (delta > 0 → eRegulon more active in activated MG)")
print(f"{'─'*70}")

if auc_m is not None and act_auc is not None and rest_auc is not None:
    # Build per-TF mean AUC (averaging across all eRegulon columns for that TF)
    auc_rows = []
    for tf in reliable_tfs:
        matches = [c for c in auc_m.var_names if c.startswith(tf + "_")]
        if not matches:
            continue
        act_mean  = act_auc[matches].mean(axis=1).values
        rest_mean = rest_auc[matches].mean(axis=1).values
        delta     = act_mean.mean() - rest_mean.mean()
        _, pval   = stats.mannwhitneyu(act_mean, rest_mean, alternative="two-sided")
        n_tgts    = int(ereg_sizes.get(tf, 0))
        rna_fc    = da_rna.loc[tf, "avg_log2FC"] if tf in da_rna.index else np.nan
        auc_rows.append(dict(
            TF=tf, delta_AUC=round(delta, 4), p_val=round(pval, 6),
            act_mean_AUC=round(act_mean.mean(), 4),
            rest_mean_AUC=round(rest_mean.mean(), 4),
            n_ereg_targets=n_tgts, TF_RNA_log2FC=round(float(rna_fc), 3)
            if not np.isnan(rna_fc) else np.nan,
            n_ereg_cols=len(matches),
        ))
    auc_df = pd.DataFrame(auc_rows)
    if len(auc_df):
        _, q, _, _ = multipletests(auc_df["p_val"], method="fdr_bh")
        auc_df["q_val"]   = q.round(6)
        auc_df["AUC_sig"] = auc_df["q_val"] < 0.05
        auc_df = auc_df.sort_values("delta_AUC", ascending=False)

        print(f"\n  {'TF':<12} {'delta_AUC':>10} {'q_val':>8} {'act_AUC':>8} "
              f"{'rest_AUC':>9} {'RNA_FC':>8} {'n_tgt':>6}  sig")
        print("  " + "─" * 75)
        for _, r in auc_df.iterrows():
            sig_s = "✓" if r["AUC_sig"] else "·"
            rfc_s = f"{r['TF_RNA_log2FC']:>+8.3f}" if not np.isnan(r["TF_RNA_log2FC"]) else "       -"
            print(f"  {r['TF']:<12} {r['delta_AUC']:>+10.4f} {r['q_val']:>8.4f} "
                  f"{r['act_mean_AUC']:>8.4f} {r['rest_mean_AUC']:>9.4f} "
                  f"{rfc_s} {int(r['n_ereg_targets']):>6}  {sig_s}")

        act_sig_tfs  = auc_df[auc_df["AUC_sig"] & (auc_df["delta_AUC"] > 0)]["TF"].tolist()
        rest_sig_tfs = auc_df[auc_df["AUC_sig"] & (auc_df["delta_AUC"] < 0)]["TF"].tolist()
        print(f"\n  Significantly higher AUC in {ACT_LABEL}:  {act_sig_tfs}")
        print(f"  Significantly higher AUC in {REST_LABEL}: {rest_sig_tfs}")

        # Paradox flags: AUC direction contradicts RNA direction
        paradoxes = auc_df[
            auc_df["AUC_sig"] &
            auc_df["TF_RNA_log2FC"].notna() &
            (auc_df["delta_AUC"].abs() >= 0.001) &
            (np.sign(auc_df["delta_AUC"]) != np.sign(auc_df["TF_RNA_log2FC"]))
        ]
        if len(paradoxes):
            print(f"\n  PARADOX FLAGS (AUC direction ≠ RNA direction):")
            for _, r in paradoxes.iterrows():
                auc_dir = ACT_LABEL if r["delta_AUC"] > 0 else REST_LABEL
                rna_dir = ACT_LABEL if r["TF_RNA_log2FC"] > 0 else REST_LABEL
                print(f"    {r['TF']:<12} AUC→{auc_dir}  RNA→{rna_dir}  "
                      f"dAUC={r['delta_AUC']:+.4f}  RNA_FC={r['TF_RNA_log2FC']:+.3f}")
            print("    Possible causes: post-transcriptional regulation, chromatin")
            print("    pre-positioning, or cofactor differences between states.")
else:
    auc_df = pd.DataFrame()
    print("  scplus MuData not available — AUC analysis skipped.")

# ════════════════════════════════════════════════════════════════════
# BLOCK 5 — eRegulon targets: state-specific sets + concordance
#
# For each state's significant TFs, pulls their eRegulon target genes
# and checks how many are concordantly DE (TF active → targets up).
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 5: STATE-SPECIFIC eRegulon TARGETS + CONCORDANCE")
print(f"  (Concordant = TF more active in state AND target gene DE in same direction)")
print(f"{'─'*70}")

if eregs is not None and len(auc_df):
    act_sig  = act_sig_tfs  if len(auc_df) else []
    rest_sig = rest_sig_tfs if len(auc_df) else []

    for state_label, tfs, fc_sign in [
        (ACT_LABEL,  act_sig,  1),
        (REST_LABEL, rest_sig, -1),
    ]:
        if not tfs:
            print(f"\n  {state_label}: no significantly enriched eRegulon TFs.")
            continue
        print(f"\n  [{state_label} eRegulon TFs: {tfs}]")
        print(f"  {'TF':<12} {'n_targets':>10} {'concordant':>11} "
              f"{'%_concordant':>13} {'top_concordant_genes'}")
        print("  " + "─" * 75)

        state_tgt_rows = []
        for tf in tfs:
            tgt_df = eregs_rel[eregs_rel["TF"] == tf][["Gene","regulation"]].copy()
            if not len(tgt_df):
                continue
            tgt_df["RNA_FC"]   = tgt_df["Gene"].map(da_rna["avg_log2FC"])
            tgt_df["RNA_padj"] = tgt_df["Gene"].map(da_rna["p_val_adj"])
            tgt_df["RNA_sig"]  = tgt_df["RNA_padj"] < 0.05
            tgt_df["concordant"] = (
                tgt_df["RNA_sig"] &
                (np.sign(tgt_df["RNA_FC"]) == fc_sign)
            )
            n_total      = len(tgt_df)
            n_concordant = tgt_df["concordant"].sum()
            pct          = n_concordant / max(n_total, 1) * 100
            top_genes    = (tgt_df[tgt_df["concordant"]]
                            .sort_values("RNA_FC", ascending=(fc_sign < 0))
                            ["Gene"].head(6).tolist())
            print(f"  {tf:<12} {n_total:>10} {n_concordant:>11} "
                  f"{pct:>12.1f}%  {', '.join(top_genes)}")
            tgt_df["TF"] = tf
            tgt_df["state"] = state_label
            state_tgt_rows.append(tgt_df)

        if state_tgt_rows:
            state_tgts = pd.concat(state_tgt_rows, ignore_index=True)
            fname = f"eRegulons_{state_label.replace(' ','_')}_targets.csv"
            state_tgts.to_csv(OUTDIR / fname, index=False)
            print(f"\n  Saved: {fname}")
else:
    print("  eRegulon data not available — skipped.")

# ════════════════════════════════════════════════════════════════════
# BLOCK 6 — region_to_gene: MG regulatory links
#
# Filters region→gene links to those targeting DE genes in each state.
# High rho + high importance = region accessibility drives gene expression.
# This is the chromatin→gene evidence layer.
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 6: REGION→GENE LINKS FOR DE GENES")
print(f"  (Links where target gene is DE in each MG state)")
print(f"  (rho filter: |rho| > {MIN_R2G_RHO} — removes near-zero correlations)")
print(f"{'─'*70}")

try:
    r2g
    r2g_available = True
except NameError:
    r2g_available = False
    print("  r2g not in scope — skipped.")

if r2g_available:
    # Filter to placed contigs only (removes scaffold-inflated importance=1.0)
    placed = ~r2g["region"].str.contains("JAENSK|scaffold|Un_", na=False, case=False)
    r2g_pl = r2g[placed].copy()

    act_genes_set  = set(da_rna[(da_rna["avg_log2FC"] >  DEG_FC_THRESH) &
                                 (da_rna["p_val_adj"] < 0.05)].index.str.upper())
    rest_genes_set = set(da_rna[(da_rna["avg_log2FC"] < -DEG_FC_THRESH) &
                                 (da_rna["p_val_adj"] < 0.05)].index.str.upper())

    for state_label, gene_set in [(ACT_LABEL, act_genes_set),
                                   (REST_LABEL, rest_genes_set)]:
        subset = r2g_pl[
            r2g_pl["target"].str.upper().isin(gene_set) &
            (r2g_pl["rho"].abs() >= MIN_R2G_RHO)
        ].copy()
        subset["RNA_FC"] = subset["target"].map(
            lambda g: da_rna.loc[g, "avg_log2FC"] if g in da_rna.index else
                      da_rna.loc[g.upper(), "avg_log2FC"]
                      if g.upper() in da_rna.index else np.nan
        )
        n_genes   = subset["target"].nunique()
        n_regions = subset["region"].nunique()
        pos_rho   = (subset["rho"] > 0).mean() * 100

        print(f"\n  {state_label}: {len(subset):,} links  |  "
              f"{n_genes} genes  |  {n_regions} regions  |  "
              f"pos_rho={pos_rho:.1f}%")

        # Top links by importance
        disp_cols = [c for c in ["target","region","importance","rho","Distance","RNA_FC"]
                     if c in subset.columns]
        print(f"\n  Top 10 by importance:")
        top10 = subset.nlargest(10, "importance")[disp_cols].copy()
        for col in top10.select_dtypes(include="object").columns:
            top10[col] = top10[col].astype(str)
        print(top10.to_string(index=False))

        # Genes with most regulatory links (hub genes in this state)
        hub = (subset.groupby("target")
               .agg(n_regions=("region","nunique"),
                    mean_imp=("importance","mean"),
                    mean_rho=("rho","mean"))
               .sort_values("n_regions", ascending=False)
               .head(10))
        hub["RNA_FC"] = hub.index.map(
            lambda g: da_rna.loc[g, "avg_log2FC"] if g in da_rna.index else np.nan)
        print(f"\n  Top 10 genes by number of regulatory regions:")
        print(hub.round(3).to_string())

        # Save
        fname = f"r2g_{state_label.replace(' ','_')}_DE_genes.csv"
        subset.to_csv(OUTDIR / fname, index=False)
        print(f"\n  Saved: {fname}")

# ════════════════════════════════════════════════════════════════════
# BLOCK 7 — Cross-layer TF evidence summary
#
# For each reliable eRegulon TF, combines all evidence layers:
#   RNA:  TF expression FC + significance
#   ATAC: TF chromatin activity FC (da_combined)
#   AUC:  eRegulon activity delta between states
#   r2g:  how many DE target genes have supporting regulatory regions
#
# Gives a single ranked table for manuscript reporting.
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 7: CROSS-LAYER TF EVIDENCE SUMMARY")
print(f"  (All evidence layers per TF in one table)")
print(f"{'─'*70}")

cross_rows = []
for tf in reliable_tfs:
    rna_r  = da_rna.loc[tf]       if tf in da_rna.index       else None
    try:
        comb_r = da_combined.loc[tf] if tf in da_combined.index else None
    except NameError:
        comb_r = None

    # AUC stats from Block 4
    auc_r = auc_df[auc_df["TF"] == tf].iloc[0] if len(auc_df) and tf in auc_df["TF"].values else None

    # Count DE target genes with r2g support
    n_r2g_act = n_r2g_rest = 0
    if r2g_available:
        tgt_genes = set(eregs_rel[eregs_rel["TF"] == tf]["Gene"].str.upper())
        n_r2g_act  = r2g_pl[r2g_pl["target"].str.upper().isin(tgt_genes) &
                             r2g_pl["target"].str.upper().isin(act_genes_set) &
                             (r2g_pl["rho"].abs() >= MIN_R2G_RHO)]["target"].nunique()
        n_r2g_rest = r2g_pl[r2g_pl["target"].str.upper().isin(tgt_genes) &
                             r2g_pl["target"].str.upper().isin(rest_genes_set) &
                             (r2g_pl["rho"].abs() >= MIN_R2G_RHO)]["target"].nunique()

    cross_rows.append(dict(
        TF             = tf,
        n_ereg_targets = int(ereg_sizes.get(tf, 0)),
        RNA_log2FC     = round(float(rna_r["avg_log2FC"]), 3)    if rna_r is not None else np.nan,
        RNA_padj       = round(float(rna_r["p_val_adj"]), 6)     if rna_r is not None else np.nan,
        RNA_sig        = bool(rna_r["p_val_adj"] < 0.05)         if rna_r is not None else False,
        ATAC_log2FC    = round(float(comb_r["avg_log2FC.activity"]), 3) if comb_r is not None else np.nan,
        ATAC_sig       = bool(comb_r["p_val_adj.activity"] < 0.05)     if comb_r is not None else False,
        delta_AUC      = float(auc_r["delta_AUC"]) if auc_r is not None else np.nan,
        AUC_q          = float(auc_r["q_val"])      if auc_r is not None else np.nan,
        AUC_sig        = bool(auc_r["AUC_sig"])     if auc_r is not None else False,
        n_r2g_act_tgts = n_r2g_act,
        n_r2g_rest_tgts= n_r2g_rest,
    ))

cross_df = pd.DataFrame(cross_rows)
# n_sig_layers: count of sig evidence layers per TF
cross_df["n_sig_layers"] = (cross_df["RNA_sig"].astype(int) +
                             cross_df["ATAC_sig"].astype(int) +
                             cross_df["AUC_sig"].astype(int))
cross_df = cross_df.sort_values("delta_AUC", ascending=False, na_position="last")

print(f"\n  {'TF':<12} {'n_tgt':>6} {'RNA_FC':>8} {'ATAC_FC':>8} "
      f"{'dAUC':>8} {'AUC_q':>7} {'r2g_act':>8} {'r2g_rst':>8} {'lyrs':>5}")
print("  " + "─" * 80)
for _, r in cross_df.iterrows():
    rfc  = f"{r['RNA_log2FC']:>+8.3f}{'†' if r['RNA_sig'] else ' '}" \
           if not np.isnan(r["RNA_log2FC"]) else "        -"
    afc  = f"{r['ATAC_log2FC']:>+8.3f}{'†' if r['ATAC_sig'] else ' '}" \
           if not np.isnan(r["ATAC_log2FC"]) else "        -"
    dauc = f"{r['delta_AUC']:>+8.4f}{'†' if r['AUC_sig'] else ' '}" \
           if not np.isnan(r["delta_AUC"]) else "        -"
    aq   = f"{r['AUC_q']:>7.4f}" if not np.isnan(r["AUC_q"]) else "      -"
    print(f"  {r['TF']:<12} {int(r['n_ereg_targets']):>6} {rfc} {afc} "
          f"{dauc} {aq} {int(r['n_r2g_act_tgts']):>8} "
          f"{int(r['n_r2g_rest_tgts']):>8} {int(r['n_sig_layers']):>5}")

print(f"\n  †=sig(padj<0.05 or q<0.05)  dAUC=act_MG−rest_MG  "
      f"r2g_act/rst=DE target genes with region→gene support  lyrs=sig layers")

# Priority candidates: sig in ≥2 layers
priority = cross_df[cross_df["n_sig_layers"] >= 2].sort_values(
    "delta_AUC", ascending=False, na_position="last")
print(f"\n  PRIORITY TFs (≥2 significant evidence layers): "
      f"{priority['TF'].tolist()}")

# ════════════════════════════════════════════════════════════════════
# BLOCK 8 — Save outputs + interpretation summary
# ════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print(f"  BLOCK 8: OUTPUTS SAVED")
print(f"{'─'*70}")

# MG specificity CSV (RNA layer)
if rows_csv:
    spec_df = (pd.DataFrame(rows_csv)
               .drop_duplicates("gene")
               .sort_values("ratio", ascending=False))
    spec_df.to_csv(OUTDIR / "MG_specificity_check.csv", index=False)
    print(f"  MG_specificity_check.csv          ({len(spec_df)} genes)")

# AUC delta table
if len(auc_df):
    auc_df.to_csv(OUTDIR / "MG_eRegulon_AUC_delta.csv", index=False)
    print(f"  MG_eRegulon_AUC_delta.csv          ({len(auc_df)} TFs)")

# Cross-layer summary
cross_df.to_csv(OUTDIR / "MG_TF_crosslayer_evidence.csv", index=False)
print(f"  MG_TF_crosslayer_evidence.csv      ({len(cross_df)} TFs)")

ACTIVATED vs RESTING MG — COMPLETE COMPARISON
  activated MG (NM) vs resting MG (SS)
  ⚠ Complete sample confounding: states are 100% separated by sample.
  Blocks 1–3 use mean_non_MG as a sample-balanced control to assess
  whether signals are MG cell-state vs NM treatment-wide effects.
  Blocks 4–7 use the multiome layers to corroborate RNA findings.

──────────────────────────────────────────────────────────────────────
  BLOCK 1: TOP 40 ACTIVATED-MG GENES BY RNA FC
  (ratio = act_MG_mean / mean_non_MG — controls for NM sample effect)
──────────────────────────────────────────────────────────────────────

  Gene                            FC   RNA_padj  act_MG rest_MG non_MG_avg   ratio  verdict
  ─────────────────────────────────────────────────────────────────────────────────────
  ENSGALG00010015567          +10.81   2.2e-171   0.333   0.000     0.006   20.3x  ✓ MG-specific
  ENSGALG00010020169           +9.74   4.0e-106   0.158   0.000     0.003   12.1x  ✓ MG-specific
  NIBAN2  

In [12]:
# ================================================================
# VOLCANO PLOT: TF/Gene differential + target lists for key TFs
# ================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
from adjustText import adjust_text

# ── Config ────────────────────────────────────────────────────────
FC_THRESH   = 0.25
PADJ_THRESH = 0.05
TFS_OF_INT  = ["MSX1", "ELK3", "CUX2"]
FIG_OUT     = OUTDIR / "volcano_TF_gene.png"

# ── 1. Build plot dataframe ───────────────────────────────────────
vdf = da_rna[["avg_log2FC", "p_val_adj"]].copy()
vdf.columns = ["log2FC", "padj"]
vdf["padj"]      = vdf["padj"].clip(lower=1e-300)
vdf["neglog10p"] = -np.log10(vdf["padj"])
vdf["gene"]      = vdf.index

def classify(row):
    sig  = row["padj"] < PADJ_THRESH
    up   = row["log2FC"] >  FC_THRESH
    down = row["log2FC"] < -FC_THRESH
    if sig and up:   return "up"
    if sig and down: return "down"
    return "ns"

vdf["class"]     = vdf.apply(classify, axis=1)
vdf["is_tf_int"] = vdf["gene"].isin(TFS_OF_INT)

# ── 2. Select genes to label ──────────────────────────────────────
# Always label: TFs of interest + all reliable eRegulon TFs
ereg_tf_labels = reliable_tfs if reliable_tfs else []

always_label = list(set(
    vdf[vdf["is_tf_int"]]["gene"].tolist() +
    [t for t in ereg_tf_labels if t in vdf["gene"].values]
))

# Fill remaining slots with top sig genes by |FC| x -log10p
sig_df = vdf[vdf["class"] != "ns"].copy()
sig_df["score"] = sig_df["log2FC"].abs() * sig_df["neglog10p"]
remaining  = max(0, 40 - len(always_label))
auto_label = (sig_df[~sig_df["gene"].isin(always_label)]
              .nlargest(remaining, "score")["gene"].tolist())

label_genes  = set(always_label + auto_label)
vdf["label"] = vdf["gene"].where(vdf["gene"].isin(label_genes), "")

print(f"Labelling {len(label_genes)} genes  "
      f"({len(always_label)} always-on: TFs of interest + eRegulon TFs)")

# ── 3. Plot ───────────────────────────────────────────────────────
palette = {"up": "#d62728", "down": "#1f77b4", "ns": "#cccccc"}
sizes   = {"up": 18,        "down": 18,         "ns": 6}
alphas  = {"up": 0.85,      "down": 0.85,        "ns": 0.3}

fig, ax = plt.subplots(figsize=(12, 9))

for cls, grp in vdf.groupby("class"):
    grp_norm = grp[~grp["is_tf_int"]]
    grp_tf   = grp[grp["is_tf_int"]]
    ax.scatter(grp_norm["log2FC"], grp_norm["neglog10p"],
               c=palette[cls], s=sizes[cls], alpha=alphas[cls],
               linewidths=0, rasterized=True, zorder=2)
    if len(grp_tf):
        ax.scatter(grp_tf["log2FC"], grp_tf["neglog10p"],
                   c=palette[cls], s=100, alpha=1.0,
                   edgecolors="black", linewidths=1.2,
                   marker="D", zorder=5)

ax.axhline(-np.log10(PADJ_THRESH), color="grey", lw=0.8, ls="--", zorder=1)
ax.axvline( FC_THRESH,  color="grey", lw=0.8, ls="--", zorder=1)
ax.axvline(-FC_THRESH,  color="grey", lw=0.8, ls="--", zorder=1)

# ── Labels ────────────────────────────────────────────────────────
texts = []
for _, row in vdf[vdf["label"] != ""].iterrows():
    is_ereg = row["gene"] in ereg_tf_labels
    is_toi  = row["is_tf_int"]
    color   = "#7f0000" if is_toi  else (
              "#cc5500" if is_ereg else (
              "#8b0000" if row["class"] == "up"   else
              "#00008b" if row["class"] == "down" else "grey"))
    size   = 9 if (is_toi or is_ereg) else 7
    weight = "bold" if (is_toi or is_ereg) else "normal"
    t = ax.text(row["log2FC"], row["neglog10p"], row["gene"],
                fontsize=size, color=color, fontweight=weight, zorder=6)
    t.set_path_effects([
        pe.Stroke(linewidth=2, foreground="white"),
        pe.Normal()
    ])
    texts.append(t)

try:
    # adjustText >= 1.0
    adjust_text(
        texts, ax=ax,
        expand=(1.3, 1.4),
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        time_lim=15,
    )
except TypeError:
    # adjustText < 1.0
    adjust_text(
        texts, ax=ax,
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        expand_text=(1.3, 1.3),
        expand_points=(1.4, 1.4),
        lim=400,
    )

n_up   = (vdf["class"] == "up").sum()
n_down = (vdf["class"] == "down").sum()
ax.text(0.97, 0.97, f"Up: {n_up}",    transform=ax.transAxes,
        ha="right", va="top", color=palette["up"],   fontsize=9, fontweight="bold")
ax.text(0.03, 0.97, f"Down: {n_down}", transform=ax.transAxes,
        ha="left",  va="top", color=palette["down"], fontsize=9, fontweight="bold")

ax.set_xlabel("Average log₂ Fold Change (activated vs resting MG)", fontsize=11)
ax.set_ylabel("−log₁₀(adjusted p-value)", fontsize=11)
ax.set_title("Differential Expression: Activated vs Resting Müller Glia\n"
             f"(|log₂FC| > {FC_THRESH}, padj < {PADJ_THRESH}  "
             f"◆ = TF of interest  orange = eRegulon TF)",
             fontsize=12)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_OUT, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_OUT}")

# ================================================================
# TARGET LISTS: MSX1, ELK3, CUX2
# ================================================================
print("\n" + "=" * 70)
print("TARGET LISTS FOR MSX1, ELK3, CUX2")
print("=" * 70)

for tf_name in TFS_OF_INT:
    print(f"\n{'─'*60}")
    print(f"  TF: {tf_name}")

    # ── eRegulon targets ─────────────────────────────────────────
    if eregs is not None:
        ereg_tgts = eregs[eregs["TF"] == tf_name].copy()
        if len(ereg_tgts):
            ereg_tgts["RNA_log2FC"] = ereg_tgts["Gene"].map(da_rna["avg_log2FC"])
            ereg_tgts["RNA_padj"]   = ereg_tgts["Gene"].map(da_rna["p_val_adj"])
            ereg_tgts["RNA_sig"]    = ereg_tgts["RNA_padj"] < PADJ_THRESH
            reg_map = {1: "act", -1: "rep", 0: "unk"}
            ereg_tgts["reg_label"]  = ereg_tgts["regulation"].map(
                lambda x: reg_map.get(int(x), "unk") if pd.notna(x) else "unk")

            # Collapse TF-region-gene triplets → one row per unique gene
            ereg_summary = (ereg_tgts
                .groupby("Gene")
                .agg(
                    reg_label  = ("reg_label",  lambda x: x.mode()[0]),
                    n_regions  = ("reg_label",  "count"),
                    RNA_log2FC = ("RNA_log2FC", "first"),
                    RNA_padj   = ("RNA_padj",   "first"),
                    RNA_sig    = ("RNA_sig",    "first"),
                )
                .reset_index()
                .sort_values(["RNA_sig", "RNA_log2FC"], ascending=[False, False])
            )

            print(f"  eRegulon targets: {len(ereg_summary)} unique genes  "
                  f"(RNA sig: {ereg_summary['RNA_sig'].sum()})")
            print(ereg_summary[["Gene", "reg_label", "n_regions",
                                 "RNA_log2FC", "RNA_padj", "RNA_sig"]]
                  .to_string(index=False))
        else:
            print("  No eRegulon targets found")
    else:
        print("  eregs not available")

    # ── tf_to_gene top 20 ─────────────────────────────────────────
    if tf2g is not None:
        t2g_tgts = tf2g[tf2g["TF"] == tf_name].nlargest(20, "importance").copy()
        if len(t2g_tgts):
            t2g_tgts["RNA_log2FC"] = t2g_tgts["target"].map(da_rna["avg_log2FC"])
            t2g_tgts["RNA_padj"]   = t2g_tgts["target"].map(da_rna["p_val_adj"])
            print(f"\n  tf_to_gene top 20 targets (by importance):")
            print(t2g_tgts[["target", "importance", "rho",
                             "RNA_log2FC", "RNA_padj"]].to_string(index=False))
        else:
            print("  No tf_to_gene targets found")
    else:
        print("  tf2g not available")

Labelling 40 genes  (18 always-on: TFs of interest + eRegulon TFs)
Saved: /mnt/c/Users/jacwe/Downloads/SAMAP/scenic_outputs/volcano_TF_gene.png

TARGET LISTS FOR MSX1, ELK3, CUX2

────────────────────────────────────────────────────────────
  TF: MSX1
  No eRegulon targets found
  No tf_to_gene targets found

────────────────────────────────────────────────────────────
  TF: ELK3
  eRegulon targets: 993 unique genes  (RNA sig: 886)
              Gene reg_label  n_regions  RNA_log2FC      RNA_padj  RNA_sig
ENSGALG00010015567       act          2   10.813383 2.171121e-171     True
           KAZALD1       act          1    8.360784 3.501020e-122     True
            LRRC15       act          1    8.339230  0.000000e+00     True
              NID2       act          1    7.934058  3.596034e-72     True
             MEOX1       act          6    7.830138  1.109465e-25     True
              SST1       act          1    7.169504 5.912889e-202     True
              WNT4       act          2

/tmp/ipykernel_451/442863793.py:135: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [13]:
# ================================================================
# VOLCANO PLOT: Highlight EGF family genes
# ================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
from adjustText import adjust_text

# ── Config ────────────────────────────────────────────────────────
FC_THRESH   = 0.25
PADJ_THRESH = 0.05
FIG_OUT     = OUTDIR / "volcano_EGF_family.png"

# ── EGF family gene list ──────────────────────────────────────────
# EGF ligands, receptors, and close pathway members
EGF_FAMILY = {
    # Ligands
    "EGF", "HBEGF", "HEBGF", "TGFA", "AREG", "EREG", "EPGN",
    "EPIREGULIN", "BTC", "NRG1", "NRG2", "NRG3", "NRG4",
    # Receptors
    "EGFR", "ERBB2", "ERBB3", "ERBB4",
    # Direct downstream
    "GRB2", "SOS1", "SOS2", "KRAS", "NRAS", "HRAS",
    "RAF1", "BRAF", "MAP2K1", "MAP2K2", "MAPK1", "MAPK3",
    "PIK3CA", "PIK3R1", "AKT1", "AKT2", "AKT3",
    "MTOR", "PTEN", "SRC", "JAK1", "JAK2", "STAT3", "STAT5A", "STAT5B",
    # Negative regulators / feedback
    "ERRFI1", "MIG6", "LRIG1", "RALT",
    # Chicken-specific aliases sometimes seen
    "HEGFL",
}

# Check which are actually in your data
egf_in_data = {g for g in EGF_FAMILY if g in da_rna.index}
print(f"EGF family genes in da_rna: {sorted(egf_in_data)}")
print(f"({len(egf_in_data)} of {len(EGF_FAMILY)} searched)")

# ── Build plot dataframe ──────────────────────────────────────────
vdf = da_rna[["avg_log2FC", "p_val_adj"]].copy()
vdf.columns = ["log2FC", "padj"]
vdf["padj"]      = vdf["padj"].clip(lower=1e-300)
vdf["neglog10p"] = -np.log10(vdf["padj"])
vdf["gene"]      = vdf.index

def classify(row):
    sig  = row["padj"] < PADJ_THRESH
    up   = row["log2FC"] >  FC_THRESH
    down = row["log2FC"] < -FC_THRESH
    if sig and up:   return "up"
    if sig and down: return "down"
    return "ns"

vdf["class"]    = vdf.apply(classify, axis=1)
vdf["is_egf"]   = vdf["gene"].isin(egf_in_data)

# ── Label selection ───────────────────────────────────────────────
# Always label all EGF family genes found in data
always_label = vdf[vdf["is_egf"]]["gene"].tolist()

sig_df = vdf[vdf["class"] != "ns"].copy()
sig_df["score"] = sig_df["log2FC"].abs() * sig_df["neglog10p"]
remaining  = max(0, 35 - len(always_label))
auto_label = (sig_df[~sig_df["gene"].isin(always_label)]
              .nlargest(remaining, "score")["gene"].tolist())

label_genes  = set(always_label + auto_label)
vdf["label"] = vdf["gene"].where(vdf["gene"].isin(label_genes), "")

print(f"\nLabelling {len(label_genes)} genes total "
      f"({len(always_label)} EGF family always-on)")

# ── Plot ──────────────────────────────────────────────────────────
palette = {"up": "#d62728", "down": "#1f77b4", "ns": "#cccccc"}
sizes   = {"up": 18,        "down": 18,         "ns": 6}
alphas  = {"up": 0.85,      "down": 0.85,        "ns": 0.3}

fig, ax = plt.subplots(figsize=(12, 9))

# Background points
for cls, grp in vdf.groupby("class"):
    grp_bg = grp[~grp["is_egf"]]
    ax.scatter(grp_bg["log2FC"], grp_bg["neglog10p"],
               c=palette[cls], s=sizes[cls], alpha=alphas[cls],
               linewidths=0, rasterized=True, zorder=2)

# EGF family on top — colour by DE direction, star marker
egf_df = vdf[vdf["is_egf"]]
for cls, grp in egf_df.groupby("class"):
    ax.scatter(grp["log2FC"], grp["neglog10p"],
               c="#e600e6",   # magenta for all EGF genes regardless of direction
               s=120, alpha=1.0,
               edgecolors="black", linewidths=1.2,
               marker="*", zorder=5)

ax.axhline(-np.log10(PADJ_THRESH), color="grey", lw=0.8, ls="--", zorder=1)
ax.axvline( FC_THRESH,  color="grey", lw=0.8, ls="--", zorder=1)
ax.axvline(-FC_THRESH,  color="grey", lw=0.8, ls="--", zorder=1)

# ── Labels ────────────────────────────────────────────────────────
texts = []
for _, row in vdf[vdf["label"] != ""].iterrows():
    is_egf = row["is_egf"]
    color  = "#9900cc" if is_egf else (
             "#8b0000" if row["class"] == "up"   else
             "#00008b" if row["class"] == "down" else "grey")
    size   = 9  if is_egf else 7
    weight = "bold" if is_egf else "normal"
    t = ax.text(row["log2FC"], row["neglog10p"], row["gene"],
                fontsize=size, color=color, fontweight=weight, zorder=6)
    t.set_path_effects([
        pe.Stroke(linewidth=2, foreground="white"),
        pe.Normal()
    ])
    texts.append(t)

try:
    adjust_text(
        texts, ax=ax,
        expand=(1.3, 1.4),
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        time_lim=15,
    )
except TypeError:
    adjust_text(
        texts, ax=ax,
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        expand_text=(1.3, 1.3),
        expand_points=(1.4, 1.4),
        lim=400,
    )

n_up   = (vdf["class"] == "up").sum()
n_down = (vdf["class"] == "down").sum()
ax.text(0.97, 0.97, f"Up: {n_up}",    transform=ax.transAxes,
        ha="right", va="top", color=palette["up"],   fontsize=9, fontweight="bold")
ax.text(0.03, 0.97, f"Down: {n_down}", transform=ax.transAxes,
        ha="left",  va="top", color=palette["down"], fontsize=9, fontweight="bold")

ax.set_xlabel("Average log₂ Fold Change (activated vs resting MG)", fontsize=11)
ax.set_ylabel("−log₁₀(adjusted p-value)", fontsize=11)
ax.set_title("Differential Expression: Activated vs Resting Müller Glia\n"
             f"(|log₂FC| > {FC_THRESH}, padj < {PADJ_THRESH}  "
             f"★ = EGF family gene)",
             fontsize=12)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_OUT, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_OUT}")

# ── EGF family gene summary table ────────────────────────────────
print("\n" + "=" * 70)
print("EGF FAMILY GENES — EXPRESSION SUMMARY")
print("=" * 70)

if egf_in_data:
    egf_stats = (da_rna.loc[list(egf_in_data), ["avg_log2FC", "p_val_adj"]]
                 .copy()
                 .sort_values("avg_log2FC", ascending=False))
    egf_stats["sig"] = egf_stats["p_val_adj"] < PADJ_THRESH
    egf_stats["direction"] = np.where(
        egf_stats["avg_log2FC"] >  FC_THRESH, "up",
        np.where(egf_stats["avg_log2FC"] < -FC_THRESH, "down", "ns")
    )
    # Add eRegulon TF regulators if available
    if eregs is not None:
        egf_stats["eReg_TFs"] = egf_stats.index.map(
            lambda g: "|".join(eregs[eregs["Gene"] == g]["TF"].unique().tolist())
        )
    if tf2g is not None:
        egf_stats["top_tf2g"] = egf_stats.index.map(
            lambda g: "|".join(
                tf2g[tf2g["target"] == g]
                .nlargest(3, "importance")["TF"].tolist()
            )
        )
    print(egf_stats.to_string())
else:
    print("  No EGF family genes found in da_rna index — "
          "check gene symbol conventions in your data")

EGF family genes in da_rna: ['AKT1', 'AKT2', 'AKT3', 'BRAF', 'BTC', 'EGF', 'EGFR', 'ERBB2', 'HBEGF', 'HRAS', 'JAK1', 'JAK2', 'KRAS', 'LRIG1', 'MAP2K1', 'MAP2K2', 'MTOR', 'NRAS', 'NRG1', 'NRG2', 'NRG3', 'NRG4', 'PIK3CA', 'PIK3R1', 'PTEN', 'RAF1', 'SOS2', 'STAT3']
(28 of 47 searched)

Labelling 35 genes total (28 EGF family always-on)


/tmp/ipykernel_451/573670814.py:152: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved: /mnt/c/Users/jacwe/Downloads/SAMAP/scenic_outputs/volcano_EGF_family.png

EGF FAMILY GENES — EXPRESSION SUMMARY
        avg_log2FC      p_val_adj    sig direction      eReg_TFs            top_tf2g
gene                                                                                
HBEGF     4.233341   0.000000e+00   True        up                    SOX8|SOX4|ELK3
BTC       1.778047   2.663412e-45   True        up                  SOX8|SOX10|FOXO1
HRAS      1.742932   0.000000e+00   True        up                    SOX2|SOX9|SOX8
JAK1      1.416588  1.845325e-250   True        up                    SOX8|SOX4|ELK3
MAP2K2    1.002039   1.377996e-66   True        up        POU4F2    POU4F2|EBF1|SOX8
ERBB2     0.938485   2.748195e-35   True        up           JUN      SOX8|SOX2|SOX9
NRAS      0.755978   6.978950e-44   True        up          ELK3      SOX8|SOX4|SOX2
LRIG1     0.576319   5.413950e-03   True        up                MEIS2|BCL11B|TBX21
PIK3CA    0.439345   7.607332e-